In [110]:
import os
import json
import base64
import numpy as np
import pandas as pd
import geopandas as gpd
from typing import Optional
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThanOrEqual
from pyiceberg.catalog import load_catalog

In [111]:
us_cbsa = gpd.read_file("/Users/houpuli/Downloads/Core_based_statistical_area_for_the_US_July_2023_4764927935501855778.geojson")
us_msa = us_cbsa[us_cbsa['CBSATYPE'] == 'Metropolitan Statistical Area']
msa_chicago = us_msa[us_msa['CBSACODE']=='16980']
msa_chicago

,OBJECTID,CBSACODE,CBSANAME,CBSATYPE,ALAND,AWATER,geometry
162,163,16980,"Chicago-Naperville-Elgin, IL-IN",Metropolitan Statistical Area,17930824264,4924526466,"POLYGON ((-87.52670 41.14923, -87.52666 41.149..."


In [112]:
msa_chicago.bounds

,minx,miny,maxx,maxy
162,-88.942152,40.736518,-86.929359,42.495645


# Query from Foursquare

In [78]:
import os
import json
import base64
import numpy as np
import geopandas as gpd
from typing import Optional
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThanOrEqual
from pyiceberg.catalog import load_catalog


def read_token(token_path: str) -> str:
    """
    Read and return the Foursquare API token from a local JSON file.

    The function supports two valid JSON formats:
    1) An object with a "token" key: {"token": "xxxxx"}
    2) A plain string containing the token: "xxxxx"

    Parameters
    ----------
    token_path : str
        Local file path to the JSON file containing the token.

    Returns
    -------
    str
        The token string extracted from the file.

    Raises
    ------
    ValueError
        If the JSON file format is invalid or does not contain a valid token.
    """
    with open(token_path, "r") as f:
        data = json.load(f)
    if isinstance(data, dict) and "token" in data:
        return data["token"]
    if isinstance(data, str):
        return data
    raise ValueError("Token JSON must be either a string or an object with a key 'token'.")


def json_str_tr(x):
    """
    Convert non-JSON-serializable Python objects (e.g., lists, dicts, arrays)
    into string representations that can be safely exported or visualized.

    Parameters
    ----------
    x : Any
        The input value to be converted.

    Returns
    -------
    str or None
        The JSON-compatible string representation of the input value.
    """
    if x is None:
        return None
    elif isinstance(x, (str, int, float, bool)):
        return str(x)
    elif isinstance(x, (list, tuple, set, np.ndarray)):
        return str(list(x))
    elif isinstance(x, dict):
        return str({k: json_str_tr(v) for k, v in x.items()})
    elif isinstance(x, (bytes, bytearray)):
        return base64.b64encode(x).decode("ascii")
    else:
        return str(x)


def query_foursquare(
    token_path: str,
    minLon: float,
    maxLon: float,
    minLat: float,
    maxLat: float,
    limit_size: Optional[int] = None,
    table_name: str = "datasets.places_os",
    uri: str = "https://catalog.h3-hub.foursquare.com/iceberg",
    warehouse: str = "places",
):
    """
    Query Points of Interest (POIs) from the Foursquare Iceberg Catalog
    within a given geographic bounding box and return results as a GeoDataFrame.

    This function connects to the Foursquare H3-Hub Iceberg dataset, applies a spatial filter
    (latitude/longitude bounds), retrieves the data as a Pandas DataFrame, converts it into
    a GeoDataFrame (EPSG:4326), and ensures all complex fields are JSON-serializable.

    Parameters
    ----------
    token_path : str
        Path to the local JSON file containing the Foursquare API token.
        The file should follow one of these formats:
        - {"token": "xxxxx"}
        - "xxxxx"
    minLon, maxLon, minLat, maxLat : float
        Geographic bounding box coordinates (in WGS84) for the query.
    limit_size : int, optional
        Maximum number of rows to return. If None (default), returns all available rows.
    table_name : str, default "datasets.places_os"
        Name of the Iceberg table to query.
    uri : str, default "https://catalog.h3-hub.foursquare.com/iceberg"
        Base URI of the Foursquare Iceberg REST catalog.
    warehouse : str, default "places"
        Name of the warehouse used by the Iceberg catalog.

    Returns
    -------
    geopandas.GeoDataFrame
        A GeoDataFrame containing POI records with WGS84 geometry.

    Examples
    --------
    Limit to 5000 rows:
    >>> gdf = query_foursquare(
    ...     token_path="~/foursquare_token.json",
    ...     minLon=-119.8694, maxLon=-119.85346,
    ...     minLat=34.40887, maxLat=34.41727,
    ...     limit_size=5000
    ... )

    Retrieve all available records (no limit):
    >>> gdf = query_foursquare(
    ...     token_path="~/foursquare_token.json",
    ...     minLon=-119.8694, maxLon=-119.85346,
    ...     minLat=34.40887, maxLat=34.41727
    ... )
    """
    # --- Load token ---
    token = read_token(os.path.expanduser(token_path))

    # --- Connect to the Iceberg catalog ---
    catalog = load_catalog(
        "default",
        **{
            "warehouse": warehouse,
            "uri": uri,
            "token": token,
            "header.content-type": "application/vnd.api+json",
            "rest-metrics-reporting-enabled": "false",
        },
    )

    # --- Load target table ---
    table = catalog.load_table(table_name)

    # --- Define spatial filter ---
    expr = And(
        And(GreaterThanOrEqual("longitude", minLon), LessThanOrEqual("longitude", maxLon)),
        And(GreaterThanOrEqual("latitude", minLat), LessThanOrEqual("latitude", maxLat)),
    )

    # --- Run query ---
    df = table.scan(row_filter=expr, limit=limit_size).to_pandas()

    # --- Convert to GeoDataFrame ---
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326",
    )

    # --- Clean up non-serializable columns ---
    for col in gdf.columns:
        if col != "geometry":
            if gdf[col].apply(lambda x: isinstance(x, (list, dict, np.ndarray, bytes, bytearray))).any():
                gdf[col] = gdf[col].apply(json_str_tr)

    return gdf


In [88]:
foursquare_places = query_foursquare(token_path="/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/foursquare_token.json",
                       minLon=-88.942152, maxLon=-86.929359,
                       minLat=40.736518, maxLat=42.495645)

In [87]:
msa_chicago.bounds

,minx,miny,maxx,maxy
162,-88.942152,40.736518,-86.929359,42.495645


In [89]:
import ast

def safe_parse(x):
    if x is None or (isinstance(x, float)):
        return None
    try:
        result = ast.literal_eval(x)
        return result[0] if isinstance(result, list) and len(result) > 0 else None
    except:
        return x

foursquare_places["cat_str"] = foursquare_places["fsq_category_labels"].apply(safe_parse)
cats = foursquare_places["cat_str"].str.split(" > ", expand=True)
foursquare_places["cat_main"] = cats[0]
foursquare_places["cat_alt"] = cats[1]

foursquare_places["fsq_category_ids"] = foursquare_places["fsq_category_ids"].apply(lambda x: ", ".join(ast.literal_eval(x)) if isinstance(x, str) and x.startswith("[") else x)
foursquare_places["fsq_category_labels"] = foursquare_places["fsq_category_labels"].apply(lambda x: ", ".join(ast.literal_eval(x)) if isinstance(x, str) and x.startswith("[") else x)
foursquare_places

,fsq_place_id,name,latitude,longitude,address,locality,region,postcode,admin_region,post_town,...,fsq_category_ids,fsq_category_labels,placemaker_url,unresolved_flags,geom,bbox,geometry,cat_str,cat_main,cat_alt
0,4e9db3d7d22d27d0e8351d66,Fabral,40.739703,-88.930549,17904 E 3100 North Rd,Gridley,IL,61744,None,None,...,4bf58dd8d48988d130941735,Landmarks and Outdoors > Structure,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAVjuOHKcGxEBEXq6YM82p,"{'xmin': '-88.9305488234632', 'ymin': '40.7397...",POINT (-88.93055 40.73970),Landmarks and Outdoors > Structure,Landmarks and Outdoors,Structure
1,10dbafa7c75f42613ee5ca7e,Schertz Aerial Service,40.740117,-88.919359,18522 E 3100 North Rd,Gridley,IL,61744,None,None,...,"63be6904847c3692a84b9b26, 4bf58dd8d48988d15b94...",Business and Professional Services > Agricultu...,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAVjrWxkqa8kBEXrwo5caw,"{'xmin': '-88.91935879979772', 'ymin': '40.740...",POINT (-88.91936 40.74012),Business and Professional Services > Agricultu...,Business and Professional Services,Agriculture and Forestry Service
2,4f5a6cfbe4b02c05daab81ec,Kerry Office,40.740154,-88.889147,Gridley Road,Gridley,IL,61744,None,None,...,4bf58dd8d48988d124941735,Business and Professional Services > Office,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAVjjnyfa800BEXr1bVsPW,"{'xmin': '-88.88914727302044', 'ymin': '40.740...",POINT (-88.88915 40.74015),Business and Professional Services > Office,Business and Professional Services,Office
3,4c0d351bc700c9b6000fa2dd,Hendren Sport Center,40.739952,-88.885236,304 W Gridley Rd,Gridley,IL,61744,None,None,...,"59d79d6b2e268052fa2a3332, 4bf58dd8d48988d16294...",Retail > Automotive Retail > Motorsports Store...,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAVjins4e8I0BEXra+BlFG,"{'xmin': '-88.8852356744324', 'ymin': '40.7399...",POINT (-88.88524 40.73995),Retail > Automotive Retail > Motorsports Store,Retail,Automotive Retail
4,5fbd3bde9a4df41493c478c6,Phillips 66,40.739863,-88.884410,212 W Gridley Rd,Gridley,IL,61744,None,None,...,4bf58dd8d48988d113951735,Travel and Transportation > Fuel Station,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAVjiaKlet10BEXrPXema0,"{'xmin': '-88.88440950930031', 'ymin': '40.739...",POINT (-88.88441 40.73986),Travel and Transportation > Fuel Station,Travel and Transportation,Fuel Station
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
735533,4e31dc5362e1d3ed651322d9,Delta Airlines Flight 2235,42.399122,-87.055664,None,None,None,None,None,None,...,4bf58dd8d48988d1f7931735,Travel and Transportation > Transport Hub > Ai...,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAVcOQAAAAAEBFMxZvV9qV,"{'xmin': '-87.0556640625', 'ymin': '42.3991221...",POINT (-87.05566 42.39912),Travel and Transportation > Transport Hub > Ai...,Travel and Transportation,Transport Hub
735534,571e318f498e0402369f94fc,Skydeck,42.488972,-87.174413,None,None,IL,None,None,None,...,4bf58dd8d48988d130941735,Landmarks and Outdoors > Structure,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAVcspls8mLEBFPpajt/sa,"{'xmin': '-87.17441339711576', 'ymin': '42.488...",POINT (-87.17441 42.48897),Landmarks and Outdoors > Structure,Landmarks and Outdoors,Structure
735535,643aa9b4a9ecd307f8865bc6,Dark Day,42.493423,-87.019878,None,None,IL,None,None,None,...,4d4b7104d754a06370d81259,Arts and Entertainment,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAVcFFrog42EBFPyh5BiwH,"{'xmin': '-87.01987803748068', 'ymin': '42.493...",POINT (-87.01988 42.49342),Arts and Entertainment,Arts and Entertainment,None
735536,55f3a920498ecab71cef7cc6,Echo Base,42.493493,-87.019960,None,None,IL,None,None,None,...,4bf58dd8d48988d1e9941735,Sports and Recreation > Snow Sports > Ski Reso...,https://foursquare.com/placemakers/review-plac...,None,AAAAAAHAVcFHBk7OmkBFPyrHU+cI,"{'xmin': '-87.01996', 'ymin': '42.493493', 'xm...",POINT (-87.01

In [90]:
foursquare_places.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_msa_poi/chicago_fsq.geojson', driver="GeoJSON")

# Query by Google Place API

In [4]:
import math
import time
import requests
import json
import warnings
from typing import Optional, Iterable, List, Dict, Any, Tuple

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box, Point
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def read_token(token_path: str) -> str:
    """
    Read and return the token from a local JSON file.

    The function supports two valid JSON formats:
    1) An object with a "token" key: {"token": "xxxxx"}
    2) A plain string containing the token: "xxxxx"

    Parameters
    ----------
    token_path : str
        Local file path to the JSON file containing the token.

    Returns
    -------
    str
        The token string extracted from the file.

    Raises
    ------
    ValueError
        If the JSON file format is invalid or does not contain a valid token.
    """
    with open(token_path, "r") as f:
        data = json.load(f)
    if isinstance(data, dict) and "token" in data:
        return data["token"]
    if isinstance(data, str):
        return data
    raise ValueError("Token JSON must be either a string or an object with a key 'token'.")


def circle_center(
    min_lon: float,
    max_lon: float,
    min_lat: float,
    max_lat: float,
    R: float,
    base: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """
    Generate a grid of circle polygons inside a geographic bounding box.
    Returns circles in EPSG:4326 (lon/lat).

    If `base` is provided, return only circles that intersect `base`
    (via spatial join, predicate='intersects').

    Parameters
    ----------
    min_lon, max_lon, min_lat, max_lat : float
        Bounding box in EPSG:4326.
    R : float
        Grid half-spacing in meters (in EPSG:3857). Spacing is 2R.
        Each circle radius is R * sqrt(2) in meters (in EPSG:3857).
    base : geopandas.GeoDataFrame | None
        A GeoDataFrame used to filter circles by intersection.
        Will be reprojected to EPSG:4326 if needed.

    Returns
    -------
    geopandas.GeoDataFrame
        Circles (filtered if base is provided), in EPSG:4326.
    """
    # -----------------------------
    # 0) Basic input validation
    # -----------------------------
    if not (isinstance(min_lon, (int, float)) and isinstance(max_lon, (int, float))
            and isinstance(min_lat, (int, float)) and isinstance(max_lat, (int, float))):
        raise TypeError("min_lon, max_lon, min_lat, max_lat must be numeric (float or int).")
    if not (max_lon > min_lon and max_lat > min_lat):
        raise ValueError("max_lon must be > min_lon and max_lat must be > min_lat.")
    if not (isinstance(R, (int, float)) and R > 0):
        raise ValueError("R must be a positive number (meters).")

    if base is not None:
        if not isinstance(base, gpd.GeoDataFrame):
            raise TypeError("base must be a GeoDataFrame or None.")
        if base.geometry is None:
            raise ValueError("base must have a valid geometry column.")
        if base.crs is None:
            raise ValueError("base.crs is None. Please set a CRS on base before using it.")

    # -----------------------------
    # 1) Build bbox in EPSG:4326
    # -----------------------------
    poly_4326 = box(min_lon, min_lat, max_lon, max_lat)
    gdf_bbox_4326 = gpd.GeoDataFrame({"name": ["bbox_4326"]}, geometry=[poly_4326], crs="EPSG:4326")

    # -----------------------------
    # 2) Reproject bbox to EPSG:3857 (meters)
    # -----------------------------
    gdf_bbox_3857 = gdf_bbox_4326.to_crs(epsg=3857)
    minx, miny, maxx, maxy = gdf_bbox_3857.total_bounds

    # -----------------------------
    # 3) Grid centers inside bbox
    # -----------------------------
    xs = np.arange(minx + R, maxx + 1e-9, 2 * R)
    ys = np.arange(miny + R, maxy + 1e-9, 2 * R)

    if xs.size == 0 or ys.size == 0:
        warnings.warn("⚠️ No grid centers generated — check if R is too large or bbox too small.")
        centers_xy = np.empty((0, 2))
    else:
        xx, yy = np.meshgrid(xs, ys)
        centers_xy = np.column_stack([xx.ravel(), yy.ravel()])

    # -----------------------------
    # 4) Build circles in 3857 -> back to 4326
    # -----------------------------
    radius_m = R
    circles_3857 = [Point(cx, cy).buffer(radius_m, resolution=64) for (cx, cy) in centers_xy]
    circles = gpd.GeoDataFrame(geometry=circles_3857, crs="EPSG:3857").to_crs(epsg=4326)

    # -----------------------------
    # 5) Optional: filter circles by intersection with base
    # -----------------------------
    base_4326 = base.to_crs("EPSG:4326")
    circles_4326 = (
        gpd.sjoin(circles, base_4326, how="inner", predicate="intersects")
            .reset_index(drop=True)
    )
    circles_4326 = circles_4326[circles.columns]
    circles_m = circles_4326.to_crs(epsg=3857)
    centroids_m = circles_m.geometry.centroid
    centroids_ll = centroids_m.to_crs(epsg=4326)
    circles_4326["center_lon"] = centroids_ll.x
    circles_4326["center_lat"] = centroids_ll.y
    
    return circles_4326

def places_nearby_grid(
    circles: gpd.GeoDataFrame,
    *,
    token: str,
    R: float,
    place_types: List[str],
    field_mask: Iterable[str],
    max_result_count: int = 20,
    sleep_sec: float = 0.1,
) -> gpd.GeoDataFrame:
    """
    Query Google Places Nearby Search for each circle center.

    Parameters
    ----------
    circles : GeoDataFrame
        Must contain columns ['center_lon', 'center_lat'] in EPSG:4326.
    token : str
        Google Places API key.
    R : float
        Search radius in meters (should match circle_center R).
    place_types : list[str]
        includedPrimaryTypes sent to Google Places.
    field_mask : iterable[str]
        Fields requested via X-Goog-FieldMask.
    max_result_count : int, default 20
        Max results per query (Google cap).
    sleep_sec : float, default 0.1
        Sleep time between requests to avoid rate limiting.

    Returns
    -------
    GeoDataFrame
        POIs returned by Google Places Nearby Search (EPSG:4326).
    """

    url = "https://places.googleapis.com/v1/places:searchNearby"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": token,
        "X-Goog-FieldMask": ",".join(field_mask),
    }

    rows: List[Dict[str, Any]] = []

    for circle_id, row in circles.iterrows():
        lat = row["center_lat"]
        lon = row["center_lon"]

        payload = {
            "locationRestriction": {
                "circle": {
                    "center": {
                        "latitude": float(lat),
                        "longitude": float(lon),
                    },
                    "radius": float(R),
                }
            },
            "includedPrimaryTypes": place_types,
            "maxResultCount": int(max_result_count),
        }

        resp = requests.post(url, headers=headers, json=payload, timeout=30)
        data = resp.json()

        places = data.get("places", [])

        for p in places:
            rows.append({
                "circle_id": circle_id,
                "id": p.get("id"),
                "name": p.get("displayName", {}).get("text"),
                "address": p.get("formattedAddress"),
                "primary_type": p.get("primaryType"),
                "lat": p.get("location", {}).get("latitude"),
                "lon": p.get("location", {}).get("longitude"),
                "business_status": p.get("businessStatus"),
            })

        time.sleep(sleep_sec)

    df = pd.DataFrame(rows)
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326",
    )

    return gdf

In [5]:
min_lon, max_lon = -118.951733, -117.412999
min_lat, max_lat = 32.750045, 34.823307

token = read_token("/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/googleplace_token.json")
R = 5000  # meters

circles = circle_center(
    min_lon=min_lon,
    max_lon=max_lon,
    min_lat=min_lat,
    max_lat=max_lat,
    R=R,
    base=msa_la
)
circles

,geometry,center_lon,center_lat
0,"POLYGON ((-118.50258 32.78781, -118.50259 32.7...",-118.547491,32.787813
1,"POLYGON ((-118.41274 32.78781, -118.41276 32.7...",-118.457660,32.787813
2,"POLYGON ((-118.32291 32.78781, -118.32293 32.7...",-118.367828,32.787813
3,"POLYGON ((-118.23308 32.78781, -118.23309 32.7...",-118.277997,32.787813
4,"POLYGON ((-118.50258 32.86330, -118.50259 32.8...",-118.547491,32.863301
...,...,...,...
260,"POLYGON ((-117.96359 34.80323, -117.96360 34.8...",-118.008502,34.803234
261,"POLYGON ((-117.87375 34.80323, -117.87377 34.8...",-117.918670,34.803234
262,"POLYGON ((-117.78392 34.80323, -117.78394 34.8...",-117.828839,34.803234
263,"POLYGON ((-117.69409 34.80323, -117.69411 34.8...",-117.739007,34.803234


## 1.shop

In [8]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "asian_grocery_store",
    "auto_parts_store",
    "bicycle_store",
    "book_store",
    "building_materials_store",
    "butcher_shop",
    "cell_phone_store",
    "clothing_store",
    "convenience_store",
    "cosmetics_store",
    "department_store",
    "discount_store",
    "discount_supermarket",
    "electronics_store",
    "farmers_market",
    "flea_market",
    "food_store",
    "furniture_store",
    "garden_center",
    "general_store",
    "gift_shop",
    "grocery_store",
    "hardware_store",
    "health_food_store",
    "home_goods_store",
    "home_improvement_store",
    "hypermarket",
    "jewelry_store",
    "liquor_store",
    "market",
    "pet_store",
    "shoe_store",
    "shopping_mall",
    "sporting_goods_store",
    "sportswear_store",
    "store",
    "supermarket",
    "tea_store",
    "thrift_store",
    "toy_store",
    "warehouse_store",
    "wholesaler",
    "womens_clothing_store",
]

google_shop_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_shop_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,21,ChIJKVqmNlJu3YARV8defwDaAzk,James H. Ackerman Native Plant Nursery,"200 Middle Ranch Rd, Avalon, CA 90704, USA",farm,33.355729,-118.437392,OPERATIONAL,POINT (-118.43739 33.35573)
1,22,ChIJUyvyVZly3YARt5DG-WbznbM,Vons,"240 Sumner Ave, Avalon, CA 90704, USA",grocery_store,33.342234,-118.327084,OPERATIONAL,POINT (-118.32708 33.34223)
2,22,ChIJ4eFn155y3YARf6nSBlYO5u0,Abe's Liquor Store,"122 Sumner Ave, Avalon, CA 90704, USA",liquor_store,33.343272,-118.326475,OPERATIONAL,POINT (-118.32648 33.34327)
3,22,ChIJjQzJ155y3YAR1cYaupOt8NY,Shades of Catalina,"116 Sumner Ave, Avalon, CA 90704, USA",store,33.343448,-118.326318,OPERATIONAL,POINT (-118.32632 33.34345)
4,22,ChIJ6ytDLpVz3YARDLF7ft1asZs,Vons Bakery,"240 Sumner Ave, Avalon, CA 90704, USA",bakery,33.342069,-118.327067,OPERATIONAL,POINT (-118.32707 33.34207)
...,...,...,...,...,...,...,...,...,...
2592,257,ChIJefgcfc1BwoARiKRFnCp_OWg,I H Only,"8747 W Ave C 2, Lancaster, CA 93536, USA",auto_parts_store,34.789407,-118.289937,CLOSED_TEMPORARILY,POINT (-118.28994 34.78941)
2593,257,ChIJBeeuLj1FwoARCa3eP96yq7g,FastPro Performance,"7316 W Ave A, Lancaster, CA 93536, USA",store,34.819553,-118.262027,OPERATIONAL,POINT (-118.26203 34.81955)
2594,258,ChIJZ9MMegBHwoARufjX04qDeAo,Earth Stone and Rock Landscape Supply,"1387 Sierra Hwy, Rosamond, CA 93560, USA",building_materials_store,34.841113,-118.160047,OPERATIONAL,POINT (-118.16005 34.84111)
2595,258,ChIJj3DjZQBHwoARP98G4atPqqM,"C&M Topsoil, inc","1060 Sierra Hwy, Rosamond, CA 93560, USA",building_materials_store,34.837849,-118.157133,OPERATIONAL,POINT (-118.15713 34.83785)


In [9]:
google_shop_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_shop_5000.geojson', driver="GeoJSON")

In [ ]:
# google_shop_5000 = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_shop_5000.geojson', driver="GeoJSON")

## 2.automotive

In [10]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "car_dealer",
    "car_rental",
    "car_repair",
    "car_wash",
    "ebike_charging_station",
    "electric_vehicle_charging_station",
    "gas_station",
    "parking",
    "parking_garage",
    "parking_lot",
    "rest_stop",
    "tire_shop",
    "truck_dealer",
]

google_automotive_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_automotive_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,22,ChIJ2VvvK4xy3YARFZspg5WuhrM,Electric Vehicle Charging Station,"410 Avalon Cyn Rd, Avalon, CA 90704, USA",electric_vehicle_charging_station,33.339858,-118.328966,OPERATIONAL,POINT (-118.32897 33.33986)
1,23,ChIJX6QzILty3YARsnHmiRpR9Fw,Santa Catalina Island Gas Station,"Pebbly Beach Rd, Avalon, CA 90704, USA",gas_station,33.335206,-118.311799,OPERATIONAL,POINT (-118.31180 33.33521)
2,23,ChIJX6QzILty3YARCbp1XWak4zg,Sonlight Auto,"Pebbly Beach Rd, Avalon, CA 90704, USA",car_repair,33.331990,-118.310062,OPERATIONAL,POINT (-118.31006 33.33199)
3,23,ChIJE2xZ27ly3YAREjuKuQ75vfc,Electric Vehicle Charging Station,"1 Pebbly Beach Rd, Avalon, CA 90704, USA",electric_vehicle_charging_station,33.334863,-118.311751,OPERATIONAL,POINT (-118.31175 33.33486)
4,28,ChIJ70LrNgBv3YARdKPC7ob-hbQ,Blackjack Shade Structure,"Avalon, CA 90704, USA",rest_stop,33.391372,-118.396269,OPERATIONAL,POINT (-118.39627 33.39137)
...,...,...,...,...,...,...,...,...,...
2405,257,ChIJVzX-YlBBwoARDFfyXM7fx3Y,A.A.mobile truck repair and welding,"8024 W Ave E, Lancaster, CA 93536, USA",car_repair,34.761559,-118.273173,OPERATIONAL,POINT (-118.27317 34.76156)
2406,257,ChIJk0creqdBwoARmuMFo_qig20,Truck Parking Club,"9810 W Ave A 10, Lancaster, CA 93536, USA",parking_lot,34.811675,-118.266073,OPERATIONAL,POINT (-118.26607 34.81168)
2407,258,ChIJjTAUnbVawoARyxvwFIMXxJY,E Z Auto Center,"1379 Sierra Hwy, Rosamond, CA 93560, USA",car_repair,34.840700,-118.159947,OPERATIONAL,POINT (-118.15995 34.84070)
2408,258,ChIJ-x5gzSFYwoARCnJTQW622SI,The Bumper Depot,"459 Lincoln St. W, Rosamond, CA 93560, USA",car_repair,34.827336,-118.160747,OPERATIONAL,POINT (-118.16075 34.82734)


In [11]:
google_automotive_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_automotive_5000.geojson', driver="GeoJSON")

In [ ]:
# google_automotive_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_automotive_5000.geojson', driver="GeoJSON")

## 3.business

In [25]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "business_center",
    "corporate_office",
    "coworking_space",
    "farm",
    "manufacturer",
    "ranch",
    "supplier",
    "television_studio",
]

google_business_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_business_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,21,ChIJKVqmNlJu3YARV8defwDaAzk,James H. Ackerman Native Plant Nursery,"200 Middle Ranch Rd, Avalon, CA 90704, USA",farm,33.355729,-118.437392,OPERATIONAL,POINT (-118.43739 33.35573)
1,22,ChIJUzIoNlVz3YARYMFJAQ8Pdcg,Garage Door Repair Installation Service of Avalon,"215 Sumner Ave, Avalon, CA 90704, USA",supplier,33.343202,-118.327038,OPERATIONAL,POINT (-118.32704 33.34320)
2,22,ChIJzZ_Yovlz3YAROUTMFRUNf5M,Christy Lins Accountancy,"301 Eucalyptus Ave, Avalon, CA 90704, USA",corporate_office,33.341810,-118.327494,OPERATIONAL,POINT (-118.32749 33.34181)
3,22,ChIJw-fXFZly3YAR3c8Yn99sGuI,Zim's of Catalina,"125 Catalina Ave, Avalon, CA 90704, USA",manufacturer,33.343031,-118.326135,OPERATIONAL,POINT (-118.32614 33.34303)
4,23,ChIJ6fjx88pz3YARcpimvWtuM90,Water Tower,"8MQH+JQ, Avalon, CA 90704, USA",government_office,33.339011,-118.320544,OPERATIONAL,POINT (-118.32054 33.33901)
...,...,...,...,...,...,...,...,...,...
2555,258,ChIJlR3YjadHwoARK8dQLZdrsNk,San Luis Butane Distribution Inc,"1050 Sierra Hwy, Rosamond, CA 93560, USA",supplier,34.837188,-118.155782,OPERATIONAL,POINT (-118.15578 34.83719)
2556,258,ChIJxXF1I35HwoARZ6mW20klzWY,Intoxalock Ignition Interlock,"1483 Sierra Hwy, Rosamond, CA 93560, USA",manufacturer,34.838243,-118.159434,CLOSED_TEMPORARILY,POINT (-118.15943 34.83824)
2557,258,ChIJQ_2MVHJHwoARZYa9jCmCLGI,W E Snyder Master Carpenter,"2024 Penny Ln, Rosamond, CA 93560, USA",manufacturer,34.844758,-118.168111,OPERATIONAL,POINT (-118.16811 34.84476)
2558,260,ChIJ6zFfN2hPwoARo7gYmrvr7sM,Rancho Los Tres,"6180 East Avenue E, Lancaster, CA 93535, USA",ranch,34.762728,-118.019466,OPERATIONAL,POINT (-118.01947 34.76273)


In [26]:
google_business_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_business_5000.geojson', driver="GeoJSON")

In [ ]:
# google_business_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_business_5000.geojson', driver="GeoJSON")

## 4.culture

In [27]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "art_gallery",
    "art_museum",
    "art_studio",
    "auditorium",
    "castle",
    "cultural_landmark",
    "fountain",
    "historical_place",
    "history_museum",
    "monument",
    "museum",
    "performing_arts_theater",
    "sculpture",
]

google_culture_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_culture_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,21,ChIJm4GlhMJn3YARRNgYm01oP5Y,Eagle's Nest,"Middle Ranch Rd, Avalon, CA 90704, USA",historical_landmark,33.358219,-118.452572,OPERATIONAL,POINT (-118.45257 33.35822)
1,22,ChIJq00t7J5y3YARt9bc7Gu5Os4,Catalina Museum For Art & History,"217 Metropole Ave, Avalon, CA 90704, USA",museum,33.343419,-118.327903,OPERATIONAL,POINT (-118.32790 33.34342)
2,22,ChIJcWUpbwBz3YAR9pYlWgg1gmw,Tom Cushing Memorial Picnic Table,"8M89+2R, Avalon, CA 90704, USA",historical_landmark,33.315073,-118.330405,OPERATIONAL,POINT (-118.33040 33.31507)
3,22,ChIJD0L3YwBz3YARUSNvi_Wv9Cc,Wrigley Field (Avalon),"Avalon, CA 90704, USA",historical_landmark,33.338503,-118.328379,OPERATIONAL,POINT (-118.32838 33.33850)
4,23,ChIJcWUpbwBz3YAR9pYlWgg1gmw,Tom Cushing Memorial Picnic Table,"8M89+2R, Avalon, CA 90704, USA",historical_landmark,33.315073,-118.330405,OPERATIONAL,POINT (-118.33040 33.31507)
...,...,...,...,...,...,...,...,...,...
1803,250,ChIJMXSfVAfu6YARP2001IhZqNs,Ridge Route Communities Museum,"3515 Park Dr, Frazier Park, CA 93225, USA",museum,34.819857,-118.944128,OPERATIONAL,POINT (-118.94413 34.81986)
1804,250,ChIJbQj4YwDv6YAR6Uw3BM_wE1o,Tejon Pass Elevation Sign,"R43F+59, Lebec, CA 93243, USA",historical_landmark,34.802949,-118.876607,OPERATIONAL,POINT (-118.87661 34.80295)
1805,250,ChIJh4hEXgDv6YARuRJiHdRyGo8,Sterling Campground Tree,"Q48C+J5, Lebec, CA 93225, USA",historical_landmark,34.766531,-118.879543,OPERATIONAL,POINT (-118.87954 34.76653)
1806,251,ChIJtYlZGQDl6YARfFNPVmMkC9A,Tejon Benchmark,"37407 Gorman Post Rd, Gorman, CA 93243, USA",historical_landmark,34.803215,-118.815607,OPERATIONAL,POINT (-118.81561 34.80321)


In [28]:
google_culture_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_culture_5000.geojson', driver="GeoJSON")

In [ ]:
# google_culture_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_culture_5000.geojson', driver="GeoJSON")

## 5.education

In [29]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "academic_department",
    "educational_institution",
    "library",
    "preschool",
    "primary_school",
    "research_institute",
    "school",
    "secondary_school",
    "university",
]

google_education_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_education_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,22,ChIJn3kiE7Zu3YARbyu6LKmE978,Avalon High School,"200 Falls Canyon Rd, Avalon, CA 90704, USA",secondary_school,33.338556,-118.332155,OPERATIONAL,POINT (-118.33216 33.33856)
1,22,ChIJRWLBzJ5y3YARI5I1XX3L6MY,Avalon Library,"215 Sumner Ave, Avalon, CA 90704, USA",library,33.343110,-118.327168,OPERATIONAL,POINT (-118.32717 33.34311)
2,22,ChIJLSX-OZBy3YARcVZ6mnSdYqg,Catalina Island Conservancy Nature Center,"1202 Avalon Cyn Rd, Avalon, CA 90704, USA",educational_institution,33.329903,-118.337298,OPERATIONAL,POINT (-118.33730 33.32990)
3,22,ChIJUdWdFJBy3YAROeEScqGbzyI,Pre School Learning Avalon Yth,"4 Bird Park Rd, Avalon, CA 90704, USA",preschool,33.335118,-118.332818,CLOSED_TEMPORARILY,POINT (-118.33282 33.33512)
4,22,ChIJcUXjW1ht3YARcTZzczqzECc,Catalina School Adventures,"231 Catalina Ave, Avalon, CA 90704, USA",educational_institution,33.342111,-118.326641,OPERATIONAL,POINT (-118.32664 33.34211)
...,...,...,...,...,...,...,...,...,...
2198,250,ChIJswPdTMTv6YARFQZEgZG1iYo,Kathleen Stokes Music Lessons,"3524 Mt Pinos Way, Frazier Park, CA 93225, USA",school,34.822331,-118.944262,OPERATIONAL,POINT (-118.94426 34.82233)
2199,251,ChIJYzo_9Yvl6YARzSL9rW4dg1Y,Gorman Elementary School,"49847 Gorman School Rd, Gorman, CA 93243, USA",primary_school,34.792766,-118.854133,OPERATIONAL,POINT (-118.85413 34.79277)
2200,256,ChIJr3eEJABrwoARCNxMKbUDdCE,Da Meet,"48799 110th St W, Lancaster, CA 93536, USA",educational_institution,34.775944,-118.325613,OPERATIONAL,POINT (-118.32561 34.77594)
2201,262,ChIJ1T4zFgCtw4AR1YYkao2523M,Chango school,"Q5G2+5V, Edwards AFB, CA 93535, USA",primary_school,34.775448,-117.847848,OPERATIONAL,POINT (-117.84785 34.77545)


In [30]:
google_education_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_education_5000.geojson', driver="GeoJSON")

In [ ]:
# google_education_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_education_5000.geojson', driver="GeoJSON")

## 6.entertainment

In [38]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types_1 = [
    "adventure_sports_center",
    "amphitheatre",
    "amusement_center",
    "amusement_park",
    "aquarium",
    "banquet_hall",
    "barbecue_area",
    "botanical_garden",
    "bowling_alley",
    "casino",
    "childrens_camp",
    "city_park",
    "comedy_club",
    "community_center",
    "concert_hall",
    "convention_center",
    "cultural_center",
    "cycling_park",
    "dance_hall",
    "dog_park",
    "event_venue",
    "ferris_wheel",
    "garden",
    "go_karting_venue",
    "hiking_area",
    "historical_landmark",
    "indoor_playground",
    "internet_cafe",
]

place_types_2 = [
    "karaoke",
    "live_music_venue",
    "marina",
    "miniature_golf_course",
    "movie_rental",
    "movie_theater",
    "national_park",
    "night_club",
    "observation_deck",
    "off_roading_area",
    "opera_house",
    "paintball_center",
    "park",
    "philharmonic_hall",
    "picnic_ground",
    "planetarium",
    "plaza",
    "roller_coaster",
    "skateboard_park",
    "state_park",
    "tourist_attraction",
    "video_arcade",
    "vineyard",
    "visitor_center",
    "water_park",
    "wedding_venue",
    "wildlife_park",
    "wildlife_refuge",
    "zoo",
]

google_entertainment_5000_a = places_nearby_grid(
    circles, token=token, R=R,
    place_types=place_types_1,
    field_mask=field_mask, sleep_sec=0.1,
)

google_entertainment_5000_b = places_nearby_grid(
    circles, token=token, R=R,
    place_types=place_types_2,
    field_mask=field_mask, sleep_sec=0.1,
)

google_entertainment_5000 = pd.concat(
    [google_entertainment_5000_a, google_entertainment_5000_b]
).drop_duplicates(subset="id").reset_index(drop=True)

google_entertainment_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,21,ChIJm4GlhMJn3YARRNgYm01oP5Y,Eagle's Nest,"Middle Ranch Rd, Avalon, CA 90704, USA",historical_landmark,33.358219,-118.452572,OPERATIONAL,POINT (-118.45257 33.35822)
1,21,ChIJWdmFukNv3YAR5oFVUWujbnU,Middle Ranch Dog Park,"Middle Ranch Rd, Avalon, CA 90704, USA",dog_park,33.355343,-118.437730,OPERATIONAL,POINT (-118.43773 33.35534)
2,22,ChIJuR3jgoxy3YARR18stlG6zqE,Wrigley Memorial and Botanic Garden,"1400 Avalon Cyn Rd, Avalon, CA 90704, USA",garden,33.326334,-118.340180,OPERATIONAL,POINT (-118.34018 33.32633)
3,22,ChIJk9xs8mBy3YARda0z0KxLYys,Garden To Sky Summit,"400 Divide Rd, Avalon, CA 90704, USA",hiking_area,33.323433,-118.349788,OPERATIONAL,POINT (-118.34979 33.32343)
4,22,ChIJg97qE1Zz3YARzizgNxkMPoQ,Hermit Gulch Lookout,"2983-3155 Divide Rd, Avalon, CA 90704, USA",hiking_area,33.329982,-118.354668,OPERATIONAL,POINT (-118.35467 33.32998)
...,...,...,...,...,...,...,...,...,...
4190,254,ChIJnxNwbS8LwoARHNGMY9Uak0k,Rx 4 Adventure,"25660 Heather Hill Ave, Lancaster, CA 93536, USA",off_roading_area,34.783868,-118.583490,OPERATIONAL,POINT (-118.58349 34.78387)
4191,257,ChIJuYe0UgAVwoARFBIzmXbpls0,Pipe inspection point,"Kern County, CA 93560, USA",park,34.834666,-118.308901,OPERATIONAL,POINT (-118.30890 34.83467)
4192,259,ChIJ02a5LGVJwoARvmJujIkWiZo,Rosamond Dry Lake,"California, USA",nature_preserve,34.838048,-118.079309,OPERATIONAL,POINT (-118.07931 34.83805)
4193,261,ChIJAQAA7LRMwoARCBrFOXx-OeM,Branch Memorial Park,"Edwards, CA 93523, USA",park,34.824436,-117.926731,OPERATIONAL,POINT (-117.92673 34.82444)


In [39]:
google_entertainment_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_entertainment_5000.geojson', driver="GeoJSON")

In [ ]:
# google_entertainment_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_entertainment_5000.geojson', driver="GeoJSON")

## 7.facilities

In [34]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "public_bath",
    "public_bathroom",
    "stable",
]

google_facilities_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_facilities_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,22,ChIJ5cRTLABz3YARbuNpjNYAXog,Restroom,"128 Sumner Ave D, Avalon, CA 90704, USA",public_bathroom,33.343053,-118.326631,OPERATIONAL,POINT (-118.32663 33.34305)
1,31,ChIJY5KvXgD13IARkOLAeVi5U0M,Public Restroom,"Calafia Beach Park, 243 Avenida Lobeiro, San C...",public_bathroom,33.405391,-117.606045,OPERATIONAL,POINT (-117.60604 33.40539)
2,31,ChIJlxVnRXD13IARgZFRYS6rmUA,Beach Restroom,"615 Avenida Victoria, San Clemente, CA 92672, USA",public_bathroom,33.419380,-117.619791,OPERATIONAL,POINT (-117.61979 33.41938)
3,31,ChIJWetfLAD13IARQQjtheM6QpI,Public Restrooms,"327 W Paseo De Cristobal, San Clemente, CA 926...",public_bathroom,33.414960,-117.615924,OPERATIONAL,POINT (-117.61592 33.41496)
4,31,ChIJcevJxpP13IAR0fJrstvMerE,Restroom,"San Clemente Pier, San Clemente, CA 92672, USA",public_bathroom,33.418286,-117.622683,OPERATIONAL,POINT (-117.62268 33.41829)
...,...,...,...,...,...,...,...,...,...
1579,245,ChIJZ1ygAM5TwoARs9TmUC35Qqk,Sweetwater Ranch and Carriage Company,"44611 70th St E, Lancaster, CA 93535, USA",pet_boarding_service,34.693992,-118.005977,OPERATIONAL,POINT (-118.00598 34.69399)
1580,247,ChIJe4oBgOupw4ARvCrOCkGrwyU,Vault toilet,"M5QG+HH, Wilsona Gardens, CA 93535, USA",public_bathroom,34.688952,-117.823542,OPERATIONAL,POINT (-117.82354 34.68895)
1581,250,ChIJa7Ztsfvv6YARSjOGAabPTLk,Vault toilet,"5301 Ralphs Ranch Rd, Lebec, CA 93243, USA",public_bathroom,34.791702,-118.871536,OPERATIONAL,POINT (-118.87154 34.79170)
1582,250,ChIJp-Wg0Fbv6YARq9GVjEYHTcM,Vault toilet,"Gold Hill Rd, Frazier Park, CA 93225, USA",public_bathroom,34.772015,-118.878905,OPERATIONAL,POINT (-118.87890 34.77202)


In [35]:
google_facilities_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_facilities_5000.geojson', driver="GeoJSON")

In [ ]:
# google_facilities_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_facilities_5000.geojson', driver="GeoJSON")

## 8.finance

In [32]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "accounting",
    "atm",
    "bank",
]

google_finance_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_finance_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,22,ChIJ2VvvK4xy3YARLCJE729g_w8,Avalon Finance Department,"410 Avalon Cyn Rd, Avalon, CA 90704, USA",local_government_office,33.340013,-118.328575,OPERATIONAL,POINT (-118.32858 33.34001)
1,32,ChIJcacZjZGK3IAR47EdDsv-Jm4,Frontwave Credit Union: Camp Pendleton - SOI,"520409 Basilone Rd, Camp Pendleton North, CA 9...",bank,33.394676,-117.513463,OPERATIONAL,POINT (-117.51346 33.39468)
2,32,ChIJqwGJ6b-K3IARSChjhOWKL7s,H&R Block,"Soi, 520407 Basilone Rd, Camp Pendleton North,...",consultant,33.394245,-117.513787,OPERATIONAL,POINT (-117.51379 33.39424)
3,32,ChIJw-LcE1n03IARxoStxoGxapQ,Citibank ATM,"2249 S El Camino Real, San Clemente, CA 92672,...",atm,33.414911,-117.600131,OPERATIONAL,POINT (-117.60013 33.41491)
4,32,ChIJMzq2uUH13IARO10xiH2RV4Y,CoinFlip Bitcoin ATM - Rocket #678 (San Clemente),"2360 S El Camino Real, San Clemente, CA 92672,...",atm,33.413279,-117.600988,OPERATIONAL,POINT (-117.60099 33.41328)
...,...,...,...,...,...,...,...,...,...
2124,250,ChIJvbKlmovv6YAR1ub0EMrsGXk,LibertyX Bitcoin ATM,"201 Frazier Mountain Park Rd, Lebec, CA 93243,...",atm,34.818252,-118.885112,OPERATIONAL,POINT (-118.88511 34.81825)
2125,250,ChIJk9e_0I7l6YARxEheNWHuWHg,ATM (Gorman 76),"49764 Gorman Post Rd, Lebec, CA 93243, USA",atm,34.796691,-118.853217,OPERATIONAL,POINT (-118.85322 34.79669)
2126,251,ChIJk9e_0I7l6YARxEheNWHuWHg,ATM (Gorman 76),"49764 Gorman Post Rd, Lebec, CA 93243, USA",atm,34.796691,-118.853217,OPERATIONAL,POINT (-118.85322 34.79669)
2127,251,ChIJdZ4zw47l6YAR1rn9WMjuV5I,ATM,"49744 Gorman Post Rd, Lebec, CA 93243, USA",atm,34.796828,-118.852534,OPERATIONAL,POINT (-118.85253 34.79683)


In [33]:
google_finance_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_finance_5000.geojson', driver="GeoJSON")

In [ ]:
# google_finance_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_finance_5000.geojson', driver="GeoJSON")

## 9.food

In [42]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types_1 = [
    "acai_shop",
    "afghani_restaurant",
    "african_restaurant",
    "american_restaurant",
    "argentinian_restaurant",
    "asian_fusion_restaurant",
    "asian_restaurant",
    "australian_restaurant",
    "austrian_restaurant",
    "bagel_shop",
    "bakery",
    "bangladeshi_restaurant",
    "bar",
    "bar_and_grill",
    "barbecue_restaurant",
    "basque_restaurant",
    "bavarian_restaurant",
    "beer_garden",
    "belgian_restaurant",
    "bistro",
]

place_types_2 = [
    "brazilian_restaurant",
    "breakfast_restaurant",
    "brewery",
    "brewpub",
    "british_restaurant",
    "brunch_restaurant",
    "buffet_restaurant",
    "burmese_restaurant",
    "burrito_restaurant",
    "cafe",
    "cafeteria",
    "cajun_restaurant",
    "cake_shop",
    "californian_restaurant",
    "cambodian_restaurant",
    "candy_store",
    "cantonese_restaurant",
    "caribbean_restaurant",
    "cat_cafe",
    "chicken_restaurant",
]

place_types_3 = [
    "chicken_wings_restaurant",
    "chilean_restaurant",
    "chinese_noodle_restaurant",
    "chinese_restaurant",
    "chocolate_factory",
    "chocolate_shop",
    "cocktail_bar",
    "coffee_roastery",
    "coffee_shop",
    "coffee_stand",
    "colombian_restaurant",
    "confectionery",
    "croatian_restaurant",
    "cuban_restaurant",
    "czech_restaurant",
    "danish_restaurant",
    "deli",
    "dessert_restaurant",
    "dessert_shop",
    "dim_sum_restaurant",
]

place_types_4 = [
    "diner",
    "dog_cafe",
    "donut_shop",
    "dumpling_restaurant",
    "dutch_restaurant",
    "eastern_european_restaurant",
    "ethiopian_restaurant",
    "european_restaurant",
    "falafel_restaurant",
    "family_restaurant",
    "fast_food_restaurant",
    "filipino_restaurant",
    "fine_dining_restaurant",
    "fish_and_chips_restaurant",
    "fondue_restaurant",
    "food_court",
    "french_restaurant",
    "fusion_restaurant",
    "gastropub",
    "german_restaurant",
]

place_types_5 = [
    "greek_restaurant",
    "gyro_restaurant",
    "halal_restaurant",
    "hamburger_restaurant",
    "hawaiian_restaurant",
    "hookah_bar",
    "hot_dog_restaurant",
    "hot_dog_stand",
    "hot_pot_restaurant",
    "hungarian_restaurant",
    "ice_cream_shop",
    "indian_restaurant",
    "indonesian_restaurant",
    "irish_pub",
    "irish_restaurant",
    "israeli_restaurant",
    "italian_restaurant",
    "japanese_curry_restaurant",
    "japanese_izakaya_restaurant",
    "japanese_restaurant",
]

place_types_6 = [
    "juice_shop",
    "kebab_shop",
    "korean_barbecue_restaurant",
    "korean_restaurant",
    "latin_american_restaurant",
    "lebanese_restaurant",
    "lounge_bar",
    "malaysian_restaurant",
    "meal_delivery",
    "meal_takeaway",
    "mediterranean_restaurant",
    "mexican_restaurant",
    "middle_eastern_restaurant",
    "mongolian_barbecue_restaurant",
    "moroccan_restaurant",
    "noodle_shop",
    "north_indian_restaurant",
    "oyster_bar_restaurant",
    "pakistani_restaurant",
    "pastry_shop",
]

place_types_7 = [
    "persian_restaurant",
    "peruvian_restaurant",
    "pizza_delivery",
    "pizza_restaurant",
    "polish_restaurant",
    "portuguese_restaurant",
    "pub",
    "ramen_restaurant",
    "restaurant",
    "romanian_restaurant",
    "russian_restaurant",
    "salad_shop",
    "sandwich_shop",
    "scandinavian_restaurant",
    "seafood_restaurant",
    "shawarma_restaurant",
    "snack_bar",
    "soul_food_restaurant",
    "soup_restaurant",
    "south_american_restaurant",
]

place_types_8 = [
    "south_indian_restaurant",
    "southwestern_us_restaurant",
    "spanish_restaurant",
    "sports_bar",
    "sri_lankan_restaurant",
    "steak_house",
    "sushi_restaurant",
    "swiss_restaurant",
    "taco_restaurant",
    "taiwanese_restaurant",
    "tapas_restaurant",
    "tea_house",
    "tex_mex_restaurant",
    "thai_restaurant",
    "tibetan_restaurant",
    "tonkatsu_restaurant",
    "turkish_restaurant",
    "ukrainian_restaurant",
    "vegan_restaurant",
    "vegetarian_restaurant",
]

place_types_9 = [
    "vietnamese_restaurant",
    "western_restaurant",
    "wine_bar",
    "winery",
    "yakiniku_restaurant",
    "yakitori_restaurant",
]

results = []
for pt in [place_types_1, place_types_2, place_types_3, place_types_4,
           place_types_5, place_types_6, place_types_7, place_types_8, place_types_9]:
    df = places_nearby_grid(
        circles, token=token, R=R,
        place_types=pt,
        field_mask=field_mask, sleep_sec=0.1,
    )
    results.append(df)

google_food_5000 = pd.concat(results).drop_duplicates(subset="id").reset_index(drop=True)
google_food_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,13,ChIJ2dN5qamF3YARMQWmjuCZOD8,Salty Crab,"Unnamed Road, CA, USA",bar_and_grill,33.002087,-118.556021,OPERATIONAL,POINT (-118.55602 33.00209)
1,22,ChIJK7ND155y3YARHHlVB_pt9bs,The Locker Room,"124 Sumner Ave, Avalon, CA 90704, USA",bar,33.343243,-118.326499,OPERATIONAL,POINT (-118.32650 33.34324)
2,22,ChIJhfZwQQBz3YARUfBeW0eRXHo,Sunset Bar and Grill,"888 Country Club Dr, Avalon, CA 90704, USA",bar,33.340472,-118.332446,OPERATIONAL,POINT (-118.33245 33.34047)
3,22,ChIJ6ytDLpVz3YARDLF7ft1asZs,Vons Bakery,"240 Sumner Ave, Avalon, CA 90704, USA",bakery,33.342069,-118.327067,OPERATIONAL,POINT (-118.32707 33.34207)
4,27,ChIJz5si5J5v3YARkt-Ky_4dfp8,Airport in the Sky Restaurant,"1 Airport Rd, Avalon, CA 90704, USA",american_restaurant,33.403060,-118.414788,OPERATIONAL,POINT (-118.41479 33.40306)
...,...,...,...,...,...,...,...,...,...
10493,229,ChIJX3D9ZgBbwoAR3OubJgrtxpY,Bánh Mì 68,"2010 W Ave J8, Lancaster, CA 93536, USA",vietnamese_restaurant,34.681478,-118.166153,OPERATIONAL,POINT (-118.16615 34.68148)
10494,230,ChIJJ7e_-UNawoARk8nf8Ip9rOY,Thief & Barrel,"311 E Avenue K 8 Ste 125, Lancaster, CA 93535,...",winery,34.668040,-118.124989,OPERATIONAL,POINT (-118.12499 34.66804)
10495,243,ChIJT5cosmZbwoARsdxtBwff1WE,Longhouse Meadery,"2330 Mall Loop Rd #108, Lancaster, CA 93536, USA",winery,34.699856,-118.172553,OPERATIONAL,POINT (-118.17255 34.69986)
10496,244,ChIJh2bXvERbwoARRqdxKsYUWjI,Stephen Hemmert Wines,"44732 Yucca Ave, Lancaster, CA 93534, USA",winery,34.696109,-118.134235,OPERATIONAL,POINT (-118.13424 34.69611)


In [43]:
google_food_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_food_5000.geojson', driver="GeoJSON")

In [ ]:
# google_food2_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_food2_5000.geojson', driver="GeoJSON")

## 10.government

In [44]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "city_hall",
    "courthouse",
    "embassy",
    "fire_station",
    "government_office",
    "local_government_office",
    "neighborhood_police_station",
    "police",
    "post_office",
]

google_government_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_government_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,22,ChIJE8Bwxp5y3YARzTrfZ_A6IyA,United States Postal Service,"118 Metropole Ave, Avalon, CA 90704, USA",post_office,33.343952,-118.326921,OPERATIONAL,POINT (-118.32692 33.34395)
1,22,ChIJBxzH_Zty3YAR_NHFExPWrKA,Avalon City Hall,"410 Avalon Cyn Rd, Avalon, CA 90704, USA",city_hall,33.340013,-118.328575,OPERATIONAL,POINT (-118.32858 33.34001)
2,22,ChIJ2VvvK4xy3YARHdiYMPE3468,Avalon Fire Department,"420 Avalon Cyn Rd, Avalon, CA 90704, USA",fire_station,33.339445,-118.329020,OPERATIONAL,POINT (-118.32902 33.33944)
3,22,ChIJX7DHzJ5y3YARtcQhjCpmAN0,Los Angeles County Sheriff - Avalon Station,"215 Sumner Ave, Avalon, CA 90704, USA",government_office,33.343110,-118.327168,OPERATIONAL,POINT (-118.32717 33.34311)
4,22,ChIJ2VvvK4xy3YARpTU-Ibbmf5E,Los Angeles County Fire Dept. Station 55,"945 Avalon Cyn Rd, Avalon, CA 90704, USA",fire_station,33.333106,-118.335500,OPERATIONAL,POINT (-118.33550 33.33311)
...,...,...,...,...,...,...,...,...,...
2070,254,ChIJrewmkCENwoARRlCv3s-XJTk,Mo's Water Station,"24715 W Ave D, Lancaster, CA 93536, USA",government_office,34.775273,-118.567171,OPERATIONAL,POINT (-118.56717 34.77527)
2071,254,ChIJM-kbEQALwoAR-JClnjHkl-g,Neenach Pump Station,"Quartz Hill, CA 93536, USA",government_office,34.788233,-118.585693,OPERATIONAL,POINT (-118.58569 34.78823)
2072,257,ChIJl67DDGxAwoARitXwmypkqwA,Sundale Mutual Water,"7337 W Ave A, Rosamond, CA 93560, USA",government_office,34.820415,-118.261365,OPERATIONAL,POINT (-118.26136 34.82041)
2073,260,ChIJ6zFfN2hPwoARo7gYmrvr7sM,Rancho Los Tres,"6180 East Avenue E, Lancaster, CA 93535, USA",ranch,34.762728,-118.019466,OPERATIONAL,POINT (-118.01947 34.76273)


In [45]:
google_government_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_government_5000.geojson', driver="GeoJSON")

In [ ]:
# google_government_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_government_5000.geojson', driver="GeoJSON")

## 11.health

In [46]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "chiropractor",
    "dental_clinic",
    "dentist",
    "doctor",
    "drugstore",
    "general_hospital",
    "hospital",
    "massage",
    "massage_spa",
    "medical_center",
    "medical_clinic",
    "medical_lab",
    "pharmacy",
    "physiotherapist",
    "sauna",
    "skin_care_clinic",
    "spa",
    "tanning_studio",
    "wellness_center",
    "yoga_studio",
]

google_health_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_health_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,22,ChIJZ2OUTppy3YARxEVjG3DrDo8,Catalina Island Health,"100 Falls Canyon Rd, Avalon, CA 90704, USA",hospital,33.339199,-118.330708,OPERATIONAL,POINT (-118.33071 33.33920)
1,22,ChIJI2xONppy3YARe0zxjJB3qmA,Avalon Clinic,"100 Falls Canyon Rd, Avalon, CA 90704, USA",medical_clinic,33.339136,-118.330760,OPERATIONAL,POINT (-118.33076 33.33914)
2,22,ChIJI8k2cJly3YARNkNTPRp-awc,Catalina Island Health Med Spa,"100 Falls Canyon Rd, Avalon, CA 90704, USA",spa,33.339187,-118.330683,OPERATIONAL,POINT (-118.33068 33.33919)
3,22,ChIJh50CuJ5z3YAREaPJnRReDiE,Gaia Studio,"210 Metropole Ave, Avalon, CA 90704, USA",yoga_studio,33.343348,-118.327583,OPERATIONAL,POINT (-118.32758 33.34335)
4,22,ChIJI2xONppy3YARUCW6cM4LBuo,Catalina Island Medical Center: Chamberlin Diane,"100 Falls Canyon Rd, Avalon, CA 90704, USA",doctor,33.339187,-118.330683,OPERATIONAL,POINT (-118.33068 33.33919)
...,...,...,...,...,...,...,...,...,...
2330,258,ChIJW72N_CdFwoAR5kmSwXd7evw,"Andrew Romany Labib Nashed, MD | Kaiser Perman...","43112 15th St, Lancaster, CA 93534, USA",doctor,34.766003,-118.166691,OPERATIONAL,POINT (-118.16669 34.76600)
2331,258,ChIJq3EuH5hFwoARU1D4Vqchs2M,"Syed Muhammad Uzair, MD | Kaiser Permanente","43112 15th St, Lancaster, CA 93534, USA",doctor,34.766003,-118.166691,OPERATIONAL,POINT (-118.16669 34.76600)
2332,258,ChIJo5HTQbtFwoARkPubNZliAs0,"George Todd Crabb, DO | Kaiser Permanente","43112 15th St, Lancaster, CA 93534, USA",doctor,34.766003,-118.166691,OPERATIONAL,POINT (-118.16669 34.76600)
2333,258,ChIJx3ZDNP1FwoAR-JmbnAIYL_M,"Mariana Bebawy, MD | Kaiser Permanente","43112 15th St, Lancaster, CA 93534, USA",doctor,34.766003,-118.166691,OPERATIONAL,POINT (-118.16669 34.76600)


In [47]:
google_health_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_health_5000.geojson', driver="GeoJSON")

In [ ]:
# google_health_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_health_5000.geojson', driver="GeoJSON")

## 12.nature

In [48]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "beach",
    "island",
    "lake",
    "mountain_peak",
    "nature_preserve",
    "river",
    "scenic_spot",
    "woods",
]

google_nature_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_nature_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,1,ChIJXd_KgN2R3YAR1YwfdYYowRM,San Clemente Island,"San Clemente Island, California, USA",island,32.902881,-118.498074,None,POINT (-118.49807 32.90288)
1,2,ChIJXd_KgN2R3YAR1YwfdYYowRM,San Clemente Island,"San Clemente Island, California, USA",island,32.902881,-118.498074,None,POINT (-118.49807 32.90288)
2,2,ChIJxUlmBBvr3YARYVa3adtet78,Balanced Rock,"Balanced Rock, California, USA",island,32.816150,-118.354518,None,POINT (-118.35452 32.81615)
3,4,ChIJXd_KgN2R3YAR1YwfdYYowRM,San Clemente Island,"San Clemente Island, California, USA",island,32.902881,-118.498074,None,POINT (-118.49807 32.90288)
4,5,ChIJXd_KgN2R3YAR1YwfdYYowRM,San Clemente Island,"San Clemente Island, California, USA",island,32.902881,-118.498074,None,POINT (-118.49807 32.90288)
...,...,...,...,...,...,...,...,...,...
3080,259,ChIJ-T8WquRFwoARWZYuumKxN38,Los Angeles County,"California, USA",lake,34.786799,-118.124292,None,POINT (-118.12429 34.78680)
3081,259,ChIJQcpPRONFwoARwlyixsY5o38,Little Piute,"Little Piute, California, USA",lake,34.787904,-118.125435,None,POINT (-118.12543 34.78790)
3082,260,ChIJ3a1TD4FLwoARaazSzwjAG8M,Buckhorn Lake,"Buckhorn Lake, California, USA",lake,34.832480,-117.975069,None,POINT (-117.97507 34.83248)
3083,261,ChIJV6eL-0qzw4ARxRk3tzQ_77w,Kern County,"California, USA",lake,34.824142,-117.922154,None,POINT (-117.92215 34.82414)


In [49]:
google_nature_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_nature_5000.geojson', driver="GeoJSON")

In [ ]:
# google_nature_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_nature_5000.geojson', driver="GeoJSON")

## 13.worship

In [52]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "buddhist_temple",
    "church",
    "hindu_temple",
    "mosque",
    "shinto_shrine",
    "synagogue",
]


google_places_worship_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_places_worship_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,22,ChIJFdnZVZhy3YARlkuxzDksC5A,Saint Catherine of Alexandria Roman Catholic C...,"800 Beacon St, Avalon, CA 90704, USA",church,33.340978,-118.325328,OPERATIONAL,POINT (-118.32533 33.34098)
1,22,ChIJG6UF55hy3YARGxz6fEMOaXk,Kingdom Hall of Jehovah's Witnesses--Avalon,"214 Clarissa Ave, Avalon, CA 90704, USA",church,33.341843,-118.325297,OPERATIONAL,POINT (-118.32530 33.34184)
2,22,ChIJNVHScJly3YAROZYl2HFdpC4,Celebrate Recovery Catalina Island,"Cafe Room, 346 Catalina Ave, Avalon, CA 90704,...",church,33.340548,-118.327634,OPERATIONAL,POINT (-118.32763 33.34055)
3,22,ChIJgd1UsJ5y3YARsTsTQzsBY6k,Avalon Community Church,"236 Metropole Ave, Avalon, CA 90704, USA",church,33.342850,-118.327893,CLOSED_TEMPORARILY,POINT (-118.32789 33.34285)
4,31,ChIJB_SrSgz03IARauPhpa6x-yI,St. Clement's By-The-Sea Episcopal Church,"202 Avenida Aragon, San Clemente, CA 92672, USA",church,33.429294,-117.624662,OPERATIONAL,POINT (-117.62466 33.42929)
...,...,...,...,...,...,...,...,...,...
2006,250,ChIJyzHz-Nfv6YARvoP9c-qUa8Q,Frazier Mountain Community Church,"700 Falcon Way, Lebec, CA 93243, USA",church,34.799584,-118.890959,OPERATIONAL,POINT (-118.89096 34.79958)
2007,250,ChIJNxKoHwTu6YARzxOs9FQ1evw,Church of Christ,"Frazier Park, CA 93225, USA",church,34.819227,-118.949809,OPERATIONAL,POINT (-118.94981 34.81923)
2008,250,ChIJ82ONbADv6YAReiro5Bc02EE,Reflexion 47,"729 Pomona Trail, Frazier Park, CA 93225, USA",church,34.822433,-118.943599,OPERATIONAL,POINT (-118.94360 34.82243)
2009,254,ChIJn2pnoqQMwoARtvExw3jPm7Y,Grace Chapel Neenach,"25649 W Ave D, Lancaster, CA 93536, USA",church,34.776131,-118.585176,OPERATIONAL,POINT (-118.58518 34.77613)


In [53]:
google_places_worship_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_places_worship_5000.geojson', driver="GeoJSON")

In [ ]:
# google_places_worship_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_places_worship_5000.geojson', driver="GeoJSON")

## 14.services

In [51]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types_1 = [
    "aircraft_rental_service",
    "association_or_organization",
    "astrologer",
    "barber_shop",
    "beautician",
    "beauty_salon",
    "body_art_service",
    "catering_service",
    "cemetery",
    "chauffeur_service",
    "child_care_agency",
    "consultant",
    "courier_service",
    "electrician",
    "employment_agency",
    "florist",
    "food_delivery",
    "foot_care",
    "funeral_home",
    "hair_care",
    "hair_salon",
]

place_types_2 = [
    "insurance_agency",
    "laundry",
    "lawyer",
    "locksmith",
    "makeup_artist",
    "marketing_consultant",
    "moving_company",
    "nail_salon",
    "non_profit_organization",
    "painter",
    "pet_boarding_service",
    "pet_care",
    "plumber",
    "psychic",
    "real_estate_agency",
    "roofing_contractor",
    "service",
    "shipping_service",
    "storage",
    "summer_camp_organizer",
    "tailor",
    "telecommunications_service_provider",
    "tour_agency",
    "tourist_information_center",
    "travel_agency",
    "veterinary_care",
]

google_services_5000_a = places_nearby_grid(
    circles, token=token, R=R,
    place_types=place_types_1,
    field_mask=field_mask, sleep_sec=0.1,
)

google_services_5000_b = places_nearby_grid(
    circles, token=token, R=R,
    place_types=place_types_2,
    field_mask=field_mask, sleep_sec=0.1,
)

google_services_5000 = pd.concat(
    [google_services_5000_a, google_services_5000_b]
).drop_duplicates(subset="id").reset_index(drop=True)

google_services_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,22,ChIJFdnZVZhy3YARlkuxzDksC5A,Saint Catherine of Alexandria Roman Catholic C...,"800 Beacon St, Avalon, CA 90704, USA",church,33.340978,-118.325328,OPERATIONAL,POINT (-118.32533 33.34098)
1,22,ChIJ0R8_zpty3YARP6dRSnZFx8Y,Catalina Country Club,"1 Country Club Dr, Avalon, CA 90704, USA",association_or_organization,33.340105,-118.329740,OPERATIONAL,POINT (-118.32974 33.34010)
2,22,ChIJE8Bwxp5y3YARzTrfZ_A6IyA,United States Postal Service,"118 Metropole Ave, Avalon, CA 90704, USA",post_office,33.343952,-118.326921,OPERATIONAL,POINT (-118.32692 33.34395)
3,22,ChIJK4FAS5ty3YARkG88JnoD8w4,Avalon Cemetery,"Cemetery Rd, Avalon, CA 90704, USA",cemetery,33.340409,-118.334282,OPERATIONAL,POINT (-118.33428 33.34041)
4,22,ChIJuWYJ6Z5y3YARkibmsNK9HDI,Catalina Hair & Nail Salon,"205 Crescent St # 120A, Avalon, CA 90704, USA",nail_salon,33.344462,-118.327289,OPERATIONAL,POINT (-118.32729 33.34446)
...,...,...,...,...,...,...,...,...,...
4925,258,ChIJCUTWMgBHwoAR4yeu3zT0E4w,Little Texas,"QVV4+JC, Rosamond, CA 93560, USA",housing_complex,34.794028,-118.143972,OPERATIONAL,POINT (-118.14397 34.79403)
4926,258,ChIJWx0MdtJAwoARWkJ8F_d_rns,George's Backhoe,"4361 W Ave A, Rosamond, CA 93560, USA",general_contractor,34.821107,-118.210092,OPERATIONAL,POINT (-118.21009 34.82111)
4927,258,ChIJrw-m1sJBwoARMkF2Gp-C6yE,Spiralmode Design Studio Inc - Antelope Acres,"8640 Ave C8 Suite #02, Lancaster, CA 93536, USA",service,34.783752,-118.215677,OPERATIONAL,POINT (-118.21568 34.78375)
4928,258,ChIJQ_2MVHJHwoARZYa9jCmCLGI,W E Snyder Master Carpenter,"2024 Penny Ln, Rosamond, CA 93560, USA",manufacturer,34.844758,-118.168111,OPERATIONAL,POINT (-118.16811 34.84476)


In [54]:
google_services_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_services_5000.geojson', driver="GeoJSON")

In [ ]:
# google_services_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_services_5000.geojson', driver="GeoJSON")

## 15.sport

In [55]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "arena",
    "athletic_field",
    "fishing_charter",
    "fishing_pier",
    "fishing_pond",
    "fitness_center",
    "golf_course",
    "gym",
    "ice_skating_rink",
    "indoor_golf_course",
    "playground",
    "race_course",
    "ski_resort",
    "sports_activity_location",
    "sports_club",
    "sports_coaching",
    "sports_complex",
    "sports_school",
    "stadium",
    "swimming_pool",
    "tennis_court",
]

google_sport_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_sport_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,21,ChIJY1sk8axv3YARe1CgMjWEaWU,Middle Ranch Reservoir,"200 Middle Ranch Rd, Avalon, CA 90704, USA",sports_activity_location,33.354235,-118.444465,OPERATIONAL,POINT (-118.44447 33.35424)
1,21,ChIJDyd5s2dv3YAR-YEBIdzUYEM,Upper Bulrush Reservoir,"Bulrush, Canyon Terrace Rd, Avalon, CA 90704, USA",sports_activity_location,33.338739,-118.418119,OPERATIONAL,POINT (-118.41812 33.33874)
2,21,ChIJnwIAsUBv3YARBTUqx8toiyk,Lower Bulrush Reservoir,"Bulrush, Canyon Terrace Rd, Avalon, CA 90704, USA",sports_activity_location,33.339997,-118.422124,OPERATIONAL,POINT (-118.42212 33.34000)
3,22,ChIJk9xs8mBy3YARda0z0KxLYys,Garden To Sky Summit,"400 Divide Rd, Avalon, CA 90704, USA",hiking_area,33.323433,-118.349788,OPERATIONAL,POINT (-118.34979 33.32343)
4,22,ChIJg97qE1Zz3YARzizgNxkMPoQ,Hermit Gulch Lookout,"2983-3155 Divide Rd, Avalon, CA 90704, USA",hiking_area,33.329982,-118.354668,OPERATIONAL,POINT (-118.35467 33.32998)
...,...,...,...,...,...,...,...,...,...
2642,253,ChIJAQAAAMDl6YAR_QtKtXVJ4Ec,High Desert Hunt Club,"30830 310TH ST, Gorman, CA 93243, USA",sports_club,34.768028,-118.669562,OPERATIONAL,POINT (-118.66956 34.76803)
2643,253,ChIJQXQdEhQLwoARAZ0hPLTdbeU,Pacific Crest Trail (Lake Hughes Crossing),"Pacific Crest Nat'l Scenic Trl, Lake Hughes, C...",hiking_area,34.775634,-118.607243,OPERATIONAL,POINT (-118.60724 34.77563)
2644,254,ChIJY0jwDgANwoARtbmrlP8CwRQ,Rancho La Ciria,"RCRM+89, Neenach, CA 93536, USA",sports_club,34.840819,-118.566556,OPERATIONAL,POINT (-118.56656 34.84082)
2645,254,ChIJnxNwbS8LwoARHNGMY9Uak0k,Rx 4 Adventure,"25660 Heather Hill Ave, Lancaster, CA 93536, USA",off_roading_area,34.783868,-118.583490,OPERATIONAL,POINT (-118.58349 34.78387)


In [56]:
google_sport_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_sport_5000.geojson', driver="GeoJSON")

In [ ]:
# google_sport_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_sport_5000.geojson', driver="GeoJSON")

## 16.transportation

In [57]:
field_mask = [
    "places.id",
    "places.displayName",
    "places.formattedAddress",
    "places.location",
    "places.primaryType",
    "places.businessStatus",
]

place_types = [
    "airport",
    "airstrip",
    "bike_sharing_station",
    "bridge",
    "bus_station",
    "bus_stop",
    "ferry_service",
    "ferry_terminal",
    "heliport",
    "international_airport",
    "light_rail_station",
    "park_and_ride",
    "subway_station",
    "taxi_service",
    "taxi_stand",
    "toll_station",
    "train_station",
    "train_ticket_office",
    "tram_stop",
    "transit_depot",
    "transit_station",
    "transit_stop",
    "transportation_service",
    "truck_stop",
]

google_transportation_5000 = places_nearby_grid(
    circles,
    token=token,
    R=R,
    place_types=place_types,
    field_mask=field_mask,
    sleep_sec=0.1,
)

google_transportation_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry
0,9,ChIJj8PB0lCP3YARLJ2AwHw4yRQ,The Original San Clemente Airfield,"Unnamed Road, CA, USA",airstrip,32.945894,-118.530552,OPERATIONAL,POINT (-118.53055 32.94589)
1,12,ChIJF7B3OKSG3YARREd9igNd1Nk,Naval Landing Field - San Clemente Island,"San Clemente Island, CA, USA",airport,33.019489,-118.584137,OPERATIONAL,POINT (-118.58414 33.01949)
2,13,ChIJF7B3OKSG3YARREd9igNd1Nk,Naval Landing Field - San Clemente Island,"San Clemente Island, CA, USA",airport,33.019489,-118.584137,OPERATIONAL,POINT (-118.58414 33.01949)
3,22,ChIJ0xYHJply3YARJTksaHC7fSM,Catalina Ave & 3rd St (Island Tour Plaza),"Avalon, CA 90704, USA",bus_stop,33.342575,-118.326097,OPERATIONAL,POINT (-118.32610 33.34257)
4,22,ChIJiel_sJ5y3YAREtS7Zmk73lg,Catalina Island Museum,"Avalon, CA 90704, USA",bus_stop,33.343570,-118.327630,OPERATIONAL,POINT (-118.32763 33.34357)
...,...,...,...,...,...,...,...,...,...
2487,250,ChIJ90s9p5bv6YAReXw01b7MuvM,Pay station,"5301 Ralphs Ranch Rd, Lebec, CA 93243, USA",toll_station,34.791710,-118.871495,OPERATIONAL,POINT (-118.87149 34.79171)
2488,252,ChIJ7wrPNe7h6YARTqayUr3iuaI,Quail Lake Sky Park,"34255 Lancaster Rd, Lancaster, CA 93536, USA",transportation_service,34.766781,-118.735253,OPERATIONAL,POINT (-118.73525 34.76678)
2489,252,ChIJBSAOr2Ph6YARqaY-v-p4Eko,Quail Lake Recreation Area-Parking Lot,"Q6CR+X47, Quartz Hill, CA 93536, USA",parking_lot,34.772413,-118.759703,OPERATIONAL,POINT (-118.75970 34.77241)
2490,252,ChIJb1kSi-Lh6YARQe2AMoE1UMM,Truck Parking Club,"33814 Ridge Rte Rd, Lancaster, CA 93536, USA",parking_lot,34.763681,-118.731231,OPERATIONAL,POINT (-118.73123 34.76368)


In [58]:
google_transportation_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/google_transportation_5000.geojson', driver="GeoJSON")

In [ ]:
# google_transportation_5000 = gpd.read_file('/Users/houpuli/Downloads/google_place_5000_bycat/google_transportation_5000.geojson', driver="GeoJSON")

In [60]:
datasets = {
    "automotive": google_automotive_5000,
    "business": google_business_5000,
    "culture": google_culture_5000,
    "education": google_education_5000,
    "entertainment": google_entertainment_5000,
    "facilities": google_facilities_5000,
    "finance": google_finance_5000,
    "food": google_food_5000,
    "government": google_government_5000,
    "health": google_health_5000,
    "nature": google_nature_5000,
    "worship": google_places_worship_5000,
    "services": google_services_5000,
    "shop": google_shop_5000,
    "sport": google_sport_5000,
    "transportation": google_transportation_5000,
}

# Add primary_cat to each GeoDataFrame
for cat, gdf in datasets.items():
    gdf["primary_cat"] = cat

google_placescat_5000 = gpd.GeoDataFrame(
    pd.concat(list(datasets.values()), ignore_index=True),
    crs=google_automotive_5000.crs
)
google_placescat_5000

,circle_id,id,name,address,primary_type,lat,lon,business_status,geometry,primary_cat
0,22,ChIJ2VvvK4xy3YARFZspg5WuhrM,Electric Vehicle Charging Station,"410 Avalon Cyn Rd, Avalon, CA 90704, USA",electric_vehicle_charging_station,33.339858,-118.328966,OPERATIONAL,POINT (-118.32897 33.33986),automotive
1,23,ChIJX6QzILty3YARsnHmiRpR9Fw,Santa Catalina Island Gas Station,"Pebbly Beach Rd, Avalon, CA 90704, USA",gas_station,33.335206,-118.311799,OPERATIONAL,POINT (-118.31180 33.33521),automotive
2,23,ChIJX6QzILty3YARCbp1XWak4zg,Sonlight Auto,"Pebbly Beach Rd, Avalon, CA 90704, USA",car_repair,33.331990,-118.310062,OPERATIONAL,POINT (-118.31006 33.33199),automotive
3,23,ChIJE2xZ27ly3YAREjuKuQ75vfc,Electric Vehicle Charging Station,"1 Pebbly Beach Rd, Avalon, CA 90704, USA",electric_vehicle_charging_station,33.334863,-118.311751,OPERATIONAL,POINT (-118.31175 33.33486),automotive
4,28,ChIJ70LrNgBv3YARdKPC7ob-hbQ,Blackjack Shade Structure,"Avalon, CA 90704, USA",rest_stop,33.391372,-118.396269,OPERATIONAL,POINT (-118.39627 33.39137),automotive
...,...,...,...,...,...,...,...,...,...,...
49554,250,ChIJ90s9p5bv6YAReXw01b7MuvM,Pay station,"5301 Ralphs Ranch Rd, Lebec, CA 93243, USA",toll_station,34.791710,-118.871495,OPERATIONAL,POINT (-118.87149 34.79171),transportation
49555,252,ChIJ7wrPNe7h6YARTqayUr3iuaI,Quail Lake Sky Park,"34255 Lancaster Rd, Lancaster, CA 93536, USA",transportation_service,34.766781,-118.735253,OPERATIONAL,POINT (-118.73525 34.76678),transportation
49556,252,ChIJBSAOr2Ph6YARqaY-v-p4Eko,Quail Lake Recreation Area-Parking Lot,"Q6CR+X47, Quartz Hill, CA 93536, USA",parking_lot,34.772413,-118.759703,OPERATIONAL,POINT (-118.75970 34.77241),transportation
49557,252,ChIJb1kSi-Lh6YARQe2AMoE1UMM,Truck Parking Club,"33814 Ridge Rte Rd, Lancaster, CA 93536, USA",parking_lot,34.763681,-118.731231,OPERATIONAL,POINT (-118.73123 34.76368),transportation


In [61]:
google_placescat_5000.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_google_place_5000_bycat/la_google_placescat_5000.geojson', driver="GeoJSON")

In [69]:
table_blist = ["administrative_area_level_3", "administrative_area_level_4", "administrative_area_level_5", "administrative_area_level_6", "administrative_area_level_7", "archipelago", "colloquial_area", "continent", "establishment", "finance", "food", "general_contractor",
"geocode", "health", "intersection", "landmark", "natural_feature", "neighborhood", "place_of_worship", "plus_code"	, "point_of_interest", "political", "postal_code_prefix", "postal_code_suffix", "postal_town", "premise", "route", "street_address", "sublocality", "sublocality_level_1",
"sublocality_level_2", "sublocality_level_3", "sublocality_level_4", "sublocality_level_5", "subpremise", "town_square"]

In [ ]:
CAT_TO_TYPES = {
    "shop": [
        "asian_grocery_store", "auto_parts_store", "bicycle_store", "book_store", "building_materials_store",
        "butcher_shop", "cell_phone_store", "clothing_store", "convenience_store", "cosmetics_store",
        "department_store", "discount_store", "discount_supermarket", "electronics_store", "farmers_market",
        "flea_market", "food_store", "furniture_store", "garden_center", "general_store", "gift_shop",
        "grocery_store", "hardware_store", "health_food_store", "home_goods_store", "home_improvement_store",
        "hypermarket", "jewelry_store", "liquor_store", "market", "pet_store", "shoe_store", "shopping_mall",
        "sporting_goods_store", "sportswear_store", "store", "supermarket", "tea_store", "thrift_store",
        "toy_store", "warehouse_store", "wholesaler", "womens_clothing_store",
    ],
    "automotive": [
        "car_dealer", "car_rental", "car_repair", "car_wash", "ebike_charging_station",
        "electric_vehicle_charging_station", "gas_station", "parking", "parking_garage", "parking_lot",
        "rest_stop", "tire_shop", "truck_dealer",
    ],
    "business": [
        "business_center", "corporate_office", "coworking_space", "farm", "manufacturer", "ranch",
        "supplier", "television_studio",
    ],
    "culture": [
        "art_gallery", "art_museum", "art_studio", "auditorium", "castle", "cultural_landmark", "fountain",
        "historical_place", "history_museum", "monument", "museum", "performing_arts_theater", "sculpture",
    ],
    "education": [
        "academic_department", "educational_institution", "library", "preschool", "primary_school",
        "research_institute", "school", "secondary_school", "university",
    ],
    "entertainment": [
        "adventure_sports_center", "amphitheatre", "amusement_center", "amusement_park", "aquarium",
        "banquet_hall", "barbecue_area", "botanical_garden", "bowling_alley", "casino", "childrens_camp",
        "city_park", "comedy_club", "community_center", "concert_hall", "convention_center", "cultural_center",
        "cycling_park", "dance_hall", "dog_park", "event_venue", "ferris_wheel", "garden", "go_karting_venue",
        "hiking_area", "historical_landmark", "indoor_playground", "internet_cafe", "karaoke", "live_music_venue",
        "marina", "miniature_golf_course", "movie_rental", "movie_theater", "national_park", "night_club",
        "observation_deck", "off_roading_area", "opera_house", "paintball_center", "park", "philharmonic_hall",
        "picnic_ground", "planetarium", "plaza", "roller_coaster", "skateboard_park", "state_park",
        "tourist_attraction", "video_arcade", "vineyard", "visitor_center", "water_park", "wedding_venue",
        "wildlife_park", "wildlife_refuge", "zoo",
    ],
    "facilities": ["public_bath", "public_bathroom", "stable"],
    "finance": ["accounting", "atm", "bank"],
    "food": [
        "acai_shop", "afghani_restaurant", "african_restaurant", "american_restaurant", "argentinian_restaurant",
        "asian_fusion_restaurant", "asian_restaurant", "australian_restaurant", "austrian_restaurant", "bagel_shop",
        "bakery", "bangladeshi_restaurant", "bar", "bar_and_grill", "barbecue_restaurant", "basque_restaurant",
        "bavarian_restaurant", "beer_garden", "belgian_restaurant", "bistro", "brazilian_restaurant",
        "breakfast_restaurant", "brewery", "brewpub", "british_restaurant", "brunch_restaurant", "buffet_restaurant",
        "burmese_restaurant", "burrito_restaurant", "cafe", "cafeteria", "cajun_restaurant", "cake_shop",
        "californian_restaurant", "cambodian_restaurant", "candy_store", "cantonese_restaurant", "caribbean_restaurant",
        "cat_cafe", "chicken_restaurant", "chicken_wings_restaurant", "chilean_restaurant", "chinese_noodle_restaurant",
        "chinese_restaurant", "chocolate_factory", "chocolate_shop", "cocktail_bar", "coffee_roastery", "coffee_shop",
        "coffee_stand", "colombian_restaurant", "confectionery", "croatian_restaurant", "cuban_restaurant",
        "czech_restaurant", "danish_restaurant", "deli", "dessert_restaurant", "dessert_shop", "dim_sum_restaurant",
        "diner", "dog_cafe", "donut_shop", "dumpling_restaurant", "dutch_restaurant", "eastern_european_restaurant",
        "ethiopian_restaurant", "european_restaurant", "falafel_restaurant", "family_restaurant", "fast_food_restaurant",
        "filipino_restaurant", "fine_dining_restaurant", "fish_and_chips_restaurant", "fondue_restaurant", "food_court",
        "french_restaurant", "fusion_restaurant", "gastropub", "german_restaurant", "greek_restaurant",
        "gyro_restaurant", "halal_restaurant", "hamburger_restaurant", "hawaiian_restaurant", "hookah_bar",
        "hot_dog_restaurant", "hot_dog_stand", "hot_pot_restaurant", "hungarian_restaurant", "ice_cream_shop",
        "indian_restaurant", "indonesian_restaurant", "irish_pub", "irish_restaurant", "israeli_restaurant",
        "italian_restaurant", "japanese_curry_restaurant", "japanese_izakaya_restaurant", "japanese_restaurant",
        "juice_shop", "kebab_shop", "korean_barbecue_restaurant", "korean_restaurant", "latin_american_restaurant",
        "lebanese_restaurant", "lounge_bar", "malaysian_restaurant", "meal_delivery", "meal_takeaway",
        "mediterranean_restaurant", "mexican_restaurant", "middle_eastern_restaurant", "mongolian_barbecue_restaurant",
        "moroccan_restaurant", "noodle_shop", "north_indian_restaurant", "oyster_bar_restaurant", "pakistani_restaurant",
        "pastry_shop", "persian_restaurant", "peruvian_restaurant", "pizza_delivery", "pizza_restaurant",
        "polish_restaurant", "portuguese_restaurant", "pub", "ramen_restaurant", "restaurant", "romanian_restaurant",
        "russian_restaurant", "salad_shop", "sandwich_shop", "scandinavian_restaurant", "seafood_restaurant",
        "shawarma_restaurant", "snack_bar", "soul_food_restaurant", "soup_restaurant", "south_american_restaurant",
        "south_indian_restaurant", "southwestern_us_restaurant", "spanish_restaurant", "sports_bar",
        "sri_lankan_restaurant", "steak_house", "sushi_restaurant", "swiss_restaurant", "taco_restaurant",
        "taiwanese_restaurant", "tapas_restaurant", "tea_house", "tex_mex_restaurant", "thai_restaurant",
        "tibetan_restaurant", "tonkatsu_restaurant", "turkish_restaurant", "ukrainian_restaurant", "vegan_restaurant",
        "vegetarian_restaurant", "vietnamese_restaurant", "western_restaurant", "wine_bar", "winery",
        "yakiniku_restaurant", "yakitori_restaurant",
    ],
    "government": [
        "city_hall", "courthouse", "embassy", "fire_station", "government_office",
        "local_government_office", "neighborhood_police_station", "police", "post_office",
    ],
    "health": [
        "chiropractor", "dental_clinic", "dentist", "doctor", "drugstore", "general_hospital", "hospital",
        "massage", "massage_spa", "medical_center", "medical_clinic", "medical_lab", "pharmacy",
        "physiotherapist", "sauna", "skin_care_clinic", "spa", "tanning_studio", "wellness_center", "yoga_studio",
    ],
    "nature": ["beach", "island", "lake", "mountain_peak", "nature_preserve", "river", "scenic_spot", "woods"],
    "worship": ["buddhist_temple", "church", "hindu_temple", "mosque", "shinto_shrine", "synagogue"],
    "services": [
        "aircraft_rental_service", "association_or_organization", "astrologer", "barber_shop", "beautician",
        "beauty_salon", "body_art_service", "catering_service", "cemetery", "chauffeur_service", "child_care_agency",
        "consultant", "courier_service", "electrician", "employment_agency", "florist", "food_delivery", "foot_care",
        "funeral_home", "hair_care", "hair_salon", "insurance_agency", "laundry", "lawyer", "locksmith",
        "makeup_artist", "marketing_consultant", "moving_company", "nail_salon", "non_profit_organization", "painter",
        "pet_boarding_service", "pet_care", "plumber", "psychic", "real_estate_agency", "roofing_contractor",
        "service", "shipping_service", "storage", "summer_camp_organizer", "tailor",
        "telecommunications_service_provider", "tour_agency", "tourist_information_center", "travel_agency",
        "veterinary_care",
    ],
    "sport": [
        "arena", "athletic_field", "fishing_charter", "fishing_pier", "fishing_pond", "fitness_center",
        "golf_course", "gym", "ice_skating_rink", "indoor_golf_course", "playground", "race_course", "ski_resort",
        "sports_activity_location", "sports_club", "sports_coaching", "sports_complex", "sports_school",
        "stadium", "swimming_pool", "tennis_court",
    ],
    "transportation": [
        "airport", "airstrip", "bike_sharing_station", "bridge", "bus_station", "bus_stop", "ferry_service",
        "ferry_terminal", "heliport", "international_airport", "light_rail_station", "park_and_ride",
        "subway_station", "taxi_service", "taxi_stand", "toll_station", "train_station", "train_ticket_office",
        "tram_stop", "transit_depot", "transit_station", "transit_stop", "transportation_service", "truck_stop",
    ],
}

df = google_placescat_5000.copy()
dup_mask = df.duplicated(subset="id", keep=False)

def is_valid_type(row, cat_to_types):
    cat = row["primary_cat"]
    ptype = row["primary_type"]
    if cat not in cat_to_types:
        return False
    return ptype in cat_to_types[cat]

df["valid_type"] = True
df.loc[dup_mask, "valid_type"] = df.loc[dup_mask].apply(is_valid_type, axis=1, cat_to_types=CAT_TO_TYPES)
df_clean = df[(~dup_mask) | (df["valid_type"])].drop(columns="valid_type")

df_clean = df_clean.drop_duplicates(subset="id", keep="first")
df_clean = df_clean[~df_clean["primary_type"].isin(table_blist)].reset_index(drop=True)

# merge with google_naics_mapping to get naics_code and naics_definition
google_naics_mapping = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/mapping_google_naics.csv')
df_clean = df_clean.dropna(subset=['primary_type'])
df_clean = df_clean.merge(google_naics_mapping[['SubCategory','naics_code','naics_definition']], left_on = 'primary_type', right_on='SubCategory', how="left")
df_clean['addr_simple'] = df_clean['address'].str.split(',', n=1).str[0]
df_clean = df_clean.drop(columns=['SubCategory']).reset_index(drop=True)
df_clean

In [83]:
df_clean.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/LA MSA/la_msa_poi/la_gplc.geojson', driver="GeoJSON")

# Google VS Overture

In [113]:
import os
import json
import base64
import numpy as np
import pandas as pd
import geopandas as gpd
from typing import Optional
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThanOrEqual
from pyiceberg.catalog import load_catalog

In [114]:
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree

def search_spatial_candidates(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    k: int = 100,
    max_dist: float = 1000, 
    id_col: str = "id",
    crs_for_distance: int = 3857,
):
    """
    Attach k nearest compared POI ids & distances to reference_gdf.

    Returns
    -------
    GeoDataFrame with two new columns:
    - cand_ids   : list of compared ids
    - cand_dist_m: list of distances (meters)
    """

    ref_proj = reference_gdf.to_crs(crs_for_distance)
    cmp_proj = compared_gdf.to_crs(crs_for_distance)

    ref_xy = np.column_stack([ref_proj.geometry.x, ref_proj.geometry.y])
    cmp_xy = np.column_stack([cmp_proj.geometry.x, cmp_proj.geometry.y])

    tree = cKDTree(cmp_xy)
    k_eff = min(k, len(compared_gdf))

    dist, idx = tree.query(ref_xy, k=k_eff)

    if k_eff == 1:
        dist = dist.reshape(-1, 1)
        idx = idx.reshape(-1, 1)

    cmp_ids = compared_gdf[id_col].to_numpy()

    cand_ids = []
    cand_dists = []

    for row_idx, row_dist in zip(idx, dist):
        ids = []
        dists = []

        for j, d in zip(row_idx, row_dist):
            if d <= max_dist:
                ids.append(cmp_ids[j])
                dists.append(d)

        cand_ids.append(ids)
        cand_dists.append(dists)

    result = reference_gdf.copy()
    result["cand_ids"] = cand_ids
    result["cand_dist_m"] = cand_dists

    return result

In [115]:
FOOD_WORDS = {
    "RESTAURANT","RESTURANT","RESTARAUNT",
    "CAFE","CAFÉ","COFFEE","BAR","BISTRO","GRILL",
    "KITCHEN","DINER","EATERY","STEAKHOUSE",
    "SEAFOOD","BUFFET","BBQ","PIZZA","SUSHI","RAMEN",
    "NOODLE","NOODLES","BURGER","BURGERS","TACO","TACOS",
    "CHICKEN","WINGS","BAKERY","DELI","DELICATESSEN",
    "COURT","FOOD","EXPRESS","HOUSE","SHOP"
}

PLACE_WORDS = {
    "HALL","CENTER","CENTRE","PLAZA","MARKET","MALL",
    "GARDEN","GARDENS","PARK","SQUARE","TOWER","TOWERS",
    "STATION","TERMINAL","BUILDING","GALLERY","THEATER","SCHOOL","COURT","INN",
    "HOTEL","MOTEL","INN","SUITES","SUITE",
    "SPA","SALON","STUDIO","CENTER","CENTRE",
    "CLUB","LOUNGE","STATION","STOP"
}

LEGAL_WORDS = {
    "LLC","INC","CORP","CORPORATION","CO","COMPANY",
    "LTD","LIMITED","GROUP","HOLDINGS","OFFICE"
}

GRAMMAR = {
    "THE","OF","AT","AND","FOR","IN","ON","BY","WITH","TO","FROM"
}

NON_PRIMARY_TOKENS = FOOD_WORDS | PLACE_WORDS | LEGAL_WORDS | GRAMMAR

In [116]:
from rapidfuzz import process, fuzz
import pandas as pd
import re
import unicodedata


def clean_name(s):
    if not isinstance(s, str):
        return ""

    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c)) # 1. unicode normalize (remove accents)
    s = s.upper() # 2. uppercase
    s = re.sub(r"\([^)]*\)", "", s) 
    s = re.sub(r"\b'S\b", "", s) # new change
    s = re.sub(r"\bS\b", "", s) # new change
    s = s.encode("ascii", errors="ignore").decode() # 4. remove emoji / non ascii
    s = re.sub(r"[^\w\s]", " ", s) # 5. replace special chars with space
    s = re.sub(r"\s+", " ", s) # 6. collapse spaces

    return s.strip()

def extract_prinmary_str(name):

    tokens = name.split()
    core = [t for t in tokens if t not in NON_PRIMARY_TOKENS]
    if len(core) == 1 and len(core[0]) < 3:
        return name
    if core:
        return " ".join(core)
    return name

def match_by_name(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    re_name_col: str = "name",
    comp_name_col: str = "name",
    comp_id: str = "id",
    comp_id_col: str="cat_main",
    threshold: int = 80,
):
    """
    Perform WRatio name matching within spatial candidates.

    Returns
    -------
    GeoDataFrame with:
    - matched_id_name
    - name_score
    """

    # clean names for matching
    id_to_name_clean = compared_gdf.set_index(comp_id)[comp_name_col].apply(clean_name).apply(extract_prinmary_str).to_dict()
    # raw names for storage
    id_to_name_raw = compared_gdf.set_index(comp_id)[comp_name_col].to_dict()
    # raw compared df category
    id_to_cat = compared_gdf.set_index(comp_id)[comp_id_col].to_dict()

    matched_ids = []
    scores = []
    loc_dists = []
    matched_names = []
    matched_cats = []

    for _, row in reference_gdf.iterrows():
        query = extract_prinmary_str(clean_name(row.get(re_name_col)))

        if not isinstance(query, str) or not row["cand_ids"]:
            matched_ids.append(pd.NA)
            scores.append(pd.NA)
            loc_dists.append(pd.NA)
            matched_names.append(pd.NA)
            matched_cats.append(pd.NA)
            continue

        cand_names = [id_to_name_clean.get(cid, "") for cid in row["cand_ids"]]

        top_matches = process.extract(
            query,
            cand_names,
            scorer=fuzz.WRatio,
            limit=5
        )

        best_score = -1
        best_pos = None

        for name, wr, pos in top_matches:

            pr = fuzz.partial_ratio(query, name)
            ts = fuzz.token_sort_ratio(query, name)
            tset = fuzz.token_set_ratio(query, name)

            combined = max(wr, pr, ts, tset)

            if combined > best_score:
                best_score = combined
                best_pos = pos

        score = best_score

        if score >= threshold:
            matched_ids.append(row["cand_ids"][best_pos])
            scores.append(score)
            loc_dists.append(row["cand_dist_m"][best_pos])
            matched_names.append(id_to_name_raw.get(row["cand_ids"][best_pos]))
            matched_cats.append(id_to_cat.get(row["cand_ids"][best_pos]))
        else:
            matched_ids.append(pd.NA)
            scores.append(score)
            loc_dists.append(pd.NA)
            matched_names.append(pd.NA)
            matched_cats.append(pd.NA)


    result = reference_gdf.copy()
    result["matched_id"] = matched_ids
    result["name_score"] = scores
    result["location_distance"] = loc_dists
    result["matched_name"] = matched_names
    result["matched_cat_main"] = matched_cats

    return result

In [117]:
import pandas as pd
import geopandas as gpd
from rapidfuzz import fuzz
import re
import unicodedata

def clean_name(s):
    if not isinstance(s, str):
        return ""

    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c)) # 1. unicode normalize (remove accents)
    s = s.upper() # 2. uppercase
    s = re.sub(r"\([^)]*\)", "", s) 
    s = s.encode("ascii", errors="ignore").decode() # 4. remove emoji / non ascii
    s = re.sub(r"[^\w\s]", " ", s) # 5. replace special chars with space
    s = re.sub(r"\s+", " ", s) # 6. collapse spaces

    return s.strip()

def address_score_check(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    addr_col_ref: str = "addr_simple",
    addr_col_cmp: str = "address",
    matched_id_col: str = "matched_id",
    id_col: str = "id",
    out_col: str = "address_score",
    out_addr_col: str = "matched_address", 
):
    """
    Compute address similarity score (0-100) for already-matched rows.

    Only rows with non-null `matched_id_col` will be scored.
    Others will have NaN.

    Returns
    -------
    GeoDataFrame with a new column `out_col`.
    """

    # map: compared id -> compared address
    id_to_addr_clean = compared_gdf.set_index(id_col)[addr_col_cmp].apply(clean_name).to_dict()
    id_to_addr_raw = compared_gdf.set_index(id_col)[addr_col_cmp].to_dict()

    scores = []
    matched_addrs = []

    for _, row in reference_gdf.iterrows():
        matched_id = row.get(matched_id_col)

        # no matched id -> no score
        if pd.isna(matched_id):
            scores.append(pd.NA)
            matched_addrs.append(pd.NA)
            continue

        addr_ref = clean_name(row.get(addr_col_ref))
        addr_cmp = id_to_addr_clean.get(matched_id)

        if isinstance(addr_cmp, str) and addr_cmp.strip():
            matched_addrs.append(id_to_addr_raw.get(matched_id))
        else:
            matched_addrs.append(pd.NA)

        # missing address on either side -> no score
        if not isinstance(addr_ref, str) or not addr_ref.strip():
            scores.append(pd.NA)
            continue
        if not isinstance(addr_cmp, str) or not addr_cmp.strip():
            scores.append(pd.NA)
            continue

        wr = fuzz.WRatio(addr_ref, addr_cmp)
        pr = fuzz.partial_ratio(addr_ref, addr_cmp)
        ts = fuzz.token_sort_ratio(addr_ref, addr_cmp)
        tset = fuzz.token_set_ratio(addr_ref, addr_cmp)

        scores.append(int(max(wr, pr, ts, tset)))

    result = reference_gdf.copy()
    result[out_col] = scores
    result[out_addr_col] = matched_addrs
    return result

In [118]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

def calculate_similarity_check(
    df, 
    cat_col_ref: str = "primary_type", 
    cat_col_cmp: str = "matched_cat_main", 
    id_col: str = "matched_id", 
    result_col: str =  "category_sim",
):

    # 1. Setup Device
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    
    # 2. Primary Gatekeeper: matched_id must be present
    mask = df[id_col].notna() & (df[id_col].astype(str).str.strip() != "")
    
    # 3. Create a helper to handle potential Nulls in text columns
    temp_df = df[mask].copy()
    
    # Identify where text is actually missing within the masked rows
    text_missing_mask = temp_df[cat_col_ref].isna() | temp_df[cat_col_cmp].isna()
    
    # Fill NaNs with empty strings just for the encoding step
    texts_1 = temp_df[cat_col_ref].fillna("").astype(str).tolist()
    texts_2 = temp_df[cat_col_cmp].fillna("").astype(str).tolist()

    print(f"Encoding {len(temp_df)} rows...")

    # 4. Batch Encoding
    emb1 = model.encode(texts_1, batch_size=256, show_progress_bar=True, convert_to_tensor=True)
    emb2 = model.encode(texts_2, batch_size=256, show_progress_bar=True, convert_to_tensor=True)

    # 5. Calculate Similarity
    scores = torch.nn.functional.cosine_similarity(emb1, emb2, dim=1).cpu().numpy()
    
    # 6. Apply NaN to rows where text was missing
    # Even if we encoded an empty string, the result isn't "real" if data was missing
    scores[text_missing_mask.values] = np.nan

    # 7. Map back to original dataframe
    df[result_col] = np.nan
    df.loc[mask, result_col] = scores
    
    return df

In [119]:
chicago_gplc = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_msa_poi/chicago_gplc.geojson')

In [120]:
chicago_ove = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_msa_poi/chicago_ove.geojson', driver='GeoJSON')

In [121]:
chicago_gplc_ove = search_spatial_candidates(reference_gdf=chicago_gplc, compared_gdf=chicago_ove, k=100)

In [122]:
chicago_gplc_ove = match_by_name(reference_gdf=chicago_gplc_ove, compared_gdf=chicago_ove, re_name_col = "name", comp_name_col = "name", threshold=80)

In [123]:
chicago_gplc_ove = address_score_check(reference_gdf=chicago_gplc_ove, compared_gdf=chicago_ove)

In [124]:
chicago_gplc_ove = calculate_similarity_check(chicago_gplc_ove, cat_col_ref = "naics_definition", cat_col_cmp = "matched_cat_main", id_col = "matched_id", result_col =  "category_sim")

Encoding 45550 rows...


Batches:   0%|          | 0/178 [00:00<?, ?it/s]

Batches:   0%|          | 0/178 [00:00<?, ?it/s]

In [125]:
# transfer the X and Y into float type and deal with the address score
cols_to_fix = ['name_score', 'location_distance', 'address_score', 'category_sim']
for col in cols_to_fix:
    chicago_gplc_ove[col] = pd.to_numeric(chicago_gplc_ove[col], errors='coerce')
chicago_gplc_ove['address_score'] = chicago_gplc_ove['address_score'].fillna(0)
chicago_gplc_ove['category_sim'] = chicago_gplc_ove['category_sim'].fillna(0)

#### Output the matched samples

In [ ]:
df = chicago_gplc_ove[chicago_gplc_ove["matched_id"].notna()].copy()

N = 1000

weights = df["primary_cat"].map(
    df["primary_cat"].value_counts(normalize=True)
)

df_sample = df.sample(
    n=N,
    weights=weights,
    random_state=42
)
df_sample[['id','name','addr_simple','naics_definition','matched_id','matched_name','matched_address','matched_cat_main','location_distance']].to_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_google_comparison/chicago_gplc_ove_1000_sample.csv', index=False)

: 

In [ ]:
chicago_gplc_ove['matched_id'].notna().sum() / len(chicago_gplc_ove)

0.8225413643325216

#### Input the matched samples

In [267]:
df_sample_check = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/ny_gplc_ove_2000_sample.csv')
df_sample_check = df_sample_check.drop(columns='location_distance')
df_sample_check['is_true'] = df_sample_check['is_true'].fillna(df_sample_check['claude_true'])
df_sample_check = df_sample_check.merge(ny_gplc_ove[['id','name_score','location_distance','address_score','category_sim']], left_on="id", right_on="id", how="left")


In [268]:
df_sample_check['is_true'].value_counts()

is_true
1.0    1760
0.0     240
Name: count, dtype: int64

In [270]:
df = df_sample_check.copy()

In [273]:
from sklearn.preprocessing import StandardScaler

X = df[['name_score', 'location_distance', 'address_score', 'category_sim']]
y = df['is_true']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [276]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.25,
    random_state=42,
    stratify=y  # keep the same proportion of True vs False in training set and test set
)

In [277]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

gb_clf = GradientBoostingClassifier()
gb_clf.fit(X_train, y_train)

gb_y_pred = gb_clf.predict(X_test)
gb_y_prob = gb_clf.predict_proba(X_test)[:, 1]

gb_cr = classification_report(y_test, gb_y_pred)
gb_auc = roc_auc_score(y_test, gb_y_prob)

print(gb_cr)
print("GradientBoosting of AUC:", gb_auc)

              precision    recall  f1-score   support

         0.0       0.87      0.80      0.83        60
         1.0       0.97      0.98      0.98       440

    accuracy                           0.96       500
   macro avg       0.92      0.89      0.91       500
weighted avg       0.96      0.96      0.96       500

GradientBoosting of AUC: 0.9865530303030303


In [280]:
mask = ny_gplc_ove['matched_id'].notnull()
df_pred = ny_gplc_ove.loc[mask].copy()

feature_cols = ['name_score', 'location_distance', 'address_score', 'category_sim']

X_new = scaler.transform(df_pred[feature_cols])
df_pred['true_match_prob'] = gb_clf.predict_proba(X_new)[:, 1]
df_pred['is_true_match'] = df_pred['true_match_prob'] >= 0.5

ny_gplc_ove.loc[mask, 'is_true_match'] = df_pred['is_true_match']
ny_gplc_ove.loc[mask, 'true_match_prob'] = df_pred['true_match_prob']

In [284]:
ny_gplc_ove['is_true_match'].value_counts()

is_true_match
True     46493
False     7145
Name: count, dtype: int64

In [340]:
ny_gplc_ove.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 65680 entries, 0 to 65679
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   circle_id          65680 non-null  int64   
 1   id                 65680 non-null  object  
 2   name               65680 non-null  object  
 3   address            65680 non-null  object  
 4   primary_type       65680 non-null  object  
 5   lat                65680 non-null  float64 
 6   lon                65680 non-null  float64 
 7   primary_cat        65680 non-null  object  
 8   geometry           65680 non-null  geometry
 9   naics_code         65680 non-null  float64 
 10  naics_definition   65680 non-null  object  
 11  addr_simple        65680 non-null  object  
 12  cand_ids           65680 non-null  object  
 13  cand_dist_m        65680 non-null  object  
 14  matched_id         53638 non-null  object  
 15  name_score         65290 non-null  float64 
 

# Google VS Safegraph

In [ ]:
chicago_sf = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_msa_poi/chicago_sf.geojson', driver="GeoJSON")

In [ ]:
chicago_gplc_sf = search_spatial_candidates(reference_gdf=chicago_gplc, compared_gdf=chicago_sf, id_col = "PLACEKEY", k=100)

In [ ]:
chicago_gplc_sf = match_by_name(reference_gdf=chicago_gplc_sf, compared_gdf=chicago_sf, re_name_col = "name", comp_name_col = "LOCATION_NAME", comp_id = "PLACEKEY",comp_id_col ="TOP_CATEGORY",  threshold=80)

In [ ]:
chicago_gplc_sf = address_score_check(reference_gdf=chicago_gplc_sf, compared_gdf=chicago_sf, addr_col_ref = "addr_simple", addr_col_cmp = "STREET_ADDRESS", id_col = "PLACEKEY",)

In [ ]:
chicago_gplc_sf = calculate_similarity_check(chicago_gplc_sf, cat_col_ref = "naics_definition", cat_col_cmp = "matched_cat_main", id_col = "matched_id", result_col =  "category_sim")

Encoding 31496 rows...


Batches:   0%|          | 0/124 [00:00<?, ?it/s]

Batches:   0%|          | 0/124 [00:00<?, ?it/s]

In [ ]:
# transfer the X and Y into float type and deal with the address score
cols_to_fix = ['name_score', 'location_distance', 'address_score', 'category_sim']
for col in cols_to_fix:
    chicago_gplc_sf[col] = pd.to_numeric(chicago_gplc_sf[col], errors='coerce')
chicago_gplc_sf['address_score'] = chicago_gplc_sf['address_score'].fillna(0)
chicago_gplc_sf['category_sim'] = chicago_gplc_sf['category_sim'].fillna(0)

#### Output the matched samples

In [ ]:
df = chicago_gplc_sf[chicago_gplc_sf["matched_id"].notna()].copy()

N = 1000

weights = df["primary_cat"].map(
    df["primary_cat"].value_counts(normalize=True)
)

df_sample = df.sample(
    n=N,
    weights=weights,
    random_state=42
)
df_sample[['id','name','addr_simple','naics_definition', 'matched_id','matched_name','matched_address','matched_cat_main','location_distance']].to_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_google_comparison/chicago_gplc_sf_1000_sample.csv', index=False)

#### Input the matched samples

In [76]:
df_sample_check = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/ny_gplc_sf_2000_sample.csv')
df_sample_check = df_sample_check.drop(columns='location_distance')
df_sample_check = df_sample_check.merge(ny_gplc_sf[['id','name_score','location_distance','address_score','category_sim']], left_on="id", right_on="id", how="left")

In [78]:
df_sample_check['is_true'].value_counts()

is_true
1    1840
0     160
Name: count, dtype: int64

In [80]:
df = df_sample_check.copy()
from sklearn.preprocessing import StandardScaler

X = df[['name_score', 'location_distance', 'address_score', 'category_sim']]
y = df['is_true']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [81]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.25,
    random_state=42,
    stratify=y  # keep the same proportion of True vs False in training set and test set
)

In [82]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

gb_clf = GradientBoostingClassifier()
gb_clf.fit(X_train, y_train)

gb_y_pred = gb_clf.predict(X_test)
gb_y_prob = gb_clf.predict_proba(X_test)[:, 1]

gb_cr = classification_report(y_test, gb_y_pred)
gb_auc = roc_auc_score(y_test, gb_y_prob)

print(gb_cr)
print("GradientBoosting of AUC:", gb_auc)

              precision    recall  f1-score   support

           0       0.74      0.72      0.73        40
           1       0.98      0.98      0.98       460

    accuracy                           0.96       500
   macro avg       0.86      0.85      0.86       500
weighted avg       0.96      0.96      0.96       500

GradientBoosting of AUC: 0.9745652173913044


In [83]:
mask = ny_gplc_sf['matched_id'].notnull()
df_pred = ny_gplc_sf.loc[mask].copy()

feature_cols = ['name_score', 'location_distance', 'address_score', 'category_sim']

X_new = scaler.transform(df_pred[feature_cols])
df_pred['true_match_prob'] = gb_clf.predict_proba(X_new)[:, 1]
df_pred['is_true_match'] = df_pred['true_match_prob'] >= 0.5

ny_gplc_sf.loc[mask, 'is_true_match'] = df_pred['is_true_match']
ny_gplc_sf.loc[mask, 'true_match_prob'] = df_pred['true_match_prob']

In [84]:
ny_gplc_sf['is_true_match'].value_counts()

is_true_match
True     46717
False     4989
Name: count, dtype: int64

In [91]:
ny_gplc_sf.drop(columns=['cand_ids','cand_dist_m']).to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/ny_gplc_sf.geojson')

# Google VS Foursquare

In [ ]:
chicago_fsq = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_msa_poi/chicago_fsq.geojson')
chicago_fsq

,fsq_place_id,name,latitude,longitude,address,locality,region,postcode,admin_region,post_town,...,fsq_category_ids,fsq_category_labels,placemaker_url,unresolved_flags,geom,bbox,cat_str,cat_main,cat_alt,geometry
0,5514e1e3498e2dd5c01df382,Spanish for Adults Online,32.980315,-118.557569,NaN,NaN,CA,90068,NaN,NaN,...,4bf58dd8d48988d13b941735,Community and Government > Education,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXaOvN0OBREBAfXry9rZd,"{'xmin': '-118.55756932823448', 'ymin': '32.98...",Community and Government > Education,Community and Government,Education,POINT (-118.55757 32.98031)
1,52a0984311d2c0b1f972ce56,SCI Old PBX Site,32.979018,-118.553763,NaN,NaN,CA,NaN,NaN,NaN,...,4bf58dd8d48988d165941735,Landmarks and Outdoors > Scenic Lookout,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXaNw28GGt0BAfVB1pgW8,"{'xmin': '-118.55376333140417', 'ymin': '32.97...",Landmarks and Outdoors > Scenic Lookout,Landmarks and Outdoors,Scenic Lookout,POINT (-118.55376 32.97902)
2,4ec3dbaeb63468c86d55f14d,The Commons,32.997933,-118.555278,NaN,Coronado,CA,NaN,NaN,NaN,...,4bf58dd8d48988d130941735,Landmarks and Outdoors > Structure,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXaOJrGs890BAf7xG7TpE,"{'xmin': '-118.55527792427516', 'ymin': '32.99...",Landmarks and Outdoors > Structure,Landmarks and Outdoors,Structure,POINT (-118.55528 32.99793)
3,4cddd5eec4f6a35deff5c56c,San Clemente Island Commons,32.998234,-118.556404,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXaOcH9dP0UBAf8YknRNN,"{'xmin': '-118.55640407587568', 'ymin': '32.99...",NaN,NaN,NaN,POINT (-118.55640 32.99823)
4,4ed314e2b634dd2994b4c7c6,Room 207,32.998383,-118.555649,NaN,NaN,NaN,NaN,NaN,NaN,...,4d954b06a243a5684965b473,Community and Government > Residential Buildin...,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXaOPwAAAAEBAf8sAAAAA,"{'xmin': '-118.55564880371094', 'ymin': '32.99...",Community and Government > Residential Buildin...,Community and Government,Residential Building,POINT (-118.55565 32.99838)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1118178,65f83a4d11744101a4361fbc,Ram Steel Fabrication Inc,34.766547,-117.880867,NaN,Lancaster,CA,93535,NaN,NaN,...,63be6904847c3692a84b9b73,Business and Professional Services > Metals Su...,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXXhgIH/h/0BBYh41FMe1,"{'xmin': '-117.88086712349467', 'ymin': '34.76...",Business and Professional Services > Metals Su...,Business and Professional Services,Metals Supplier,POINT (-117.88087 34.76655)
1118179,4f3d1779e4b085c2d961a9bc,South Gate Edwards AFB,34.787155,-117.916437,NaN,NaN,NaN,NaN,NaN,NaN,...,4bf58dd8d48988d12d941735,Landmarks and Outdoors > Monument,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXXqm5lCP5EBBZMGBfvDE,"{'xmin': '-117.91643674723997', 'ymin': '34.78...",Landmarks and Outdoors > Monument,Landmarks and Outdoors,Monument,POINT (-117.91644 34.78716)
1118180,511bc1c8e4b02d59790cb4ab,AT&T cell site LA0206,34.760030,-117.799596,NaN,Lancaster,CA,93535,NaN,NaN,...,4bf58dd8d48988d130941735,Landmarks and Outdoors > Structure,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXXMsk4VUV0BBYUitVdfl,"{'xmin': '-117.79959571857886', 'ymin': '34.76...",Landmarks and Outdoors > Structure,Landmarks and Outdoors,Structure,POINT (-117.79960 34.76003)
1118181,4e655bcae4cdf1e2c08c43a8,Rocketsite,34.817064,-117.825914,NaN,NaN,NaN,NaN,NaN,NaN,...,4bf58dd8d48988d15f941735,Landmarks and Outdoors > Field,https://foursquare.com/placemakers/review-plac...,NaN,AAAAAAHAXXTbxgzs2kBBaJWRasJB,"{'xmin': '-117.82591391813494', 'ymin': '34.81...",Landmarks and Outdoors > Field,Landmarks and Outdoors,Field,POINT (-117.82591 34.81706)


In [ ]:
chicago_gplc_fsq = search_spatial_candidates(reference_gdf=chicago_gplc, compared_gdf=chicago_fsq, id_col = "fsq_place_id", k=100)

In [ ]:
chicago_gplc_fsq = match_by_name(reference_gdf=chicago_gplc_fsq, compared_gdf=chicago_fsq, re_name_col = "name", comp_name_col = "name", comp_id = "fsq_place_id", comp_id_col ="cat_alt",  threshold=80)

In [ ]:
chicago_gplc_fsq = address_score_check(reference_gdf=chicago_gplc_fsq, compared_gdf=chicago_fsq, addr_col_ref = "addr_simple", addr_col_cmp = "address", id_col = "fsq_place_id")

In [ ]:
chicago_gplc_fsq = calculate_similarity_check(chicago_gplc_fsq, cat_col_ref = "naics_definition", cat_col_cmp = "matched_cat_main", id_col = "matched_id", result_col =  "category_sim")

Encoding 33015 rows...


Batches:   0%|          | 0/129 [00:00<?, ?it/s]

Batches:   0%|          | 0/129 [00:00<?, ?it/s]

In [ ]:
# transfer the X and Y into float type and deal with the address score
cols_to_fix = ['name_score', 'location_distance', 'address_score', 'category_sim']
for col in cols_to_fix:
    chicago_gplc_fsq[col] = pd.to_numeric(chicago_gplc_fsq[col], errors='coerce')
chicago_gplc_fsq['address_score'] = chicago_gplc_fsq['address_score'].fillna(0)
chicago_gplc_fsq['category_sim'] = chicago_gplc_fsq['category_sim'].fillna(0)

In [ ]:
df = chicago_gplc_fsq[chicago_gplc_fsq["matched_id"].notna()].copy()

N = 1000

weights = df["primary_cat"].map(
    df["primary_cat"].value_counts(normalize=True)
)

df_sample = df.sample(
    n=N,
    weights=weights,
    random_state=42
)
df_sample[['id','name','addr_simple','naics_definition','matched_id','matched_name','matched_address','matched_cat_main','location_distance']].to_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_google_comparison/chicago_gplc_fsq_1000_sample.csv', index=False)

In [162]:
df_sample_check = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/ny_gplc_fsq_2000_sample.csv')
df_sample_check = df_sample_check.drop(columns='location_distance')
df_sample_check = df_sample_check.merge(ny_gplc_fsq[['id','name_score','location_distance','address_score','category_sim']], left_on="id", right_on="id", how="left")

In [163]:
df_sample_check['is_true'].value_counts()

is_true
1    1643
0     357
Name: count, dtype: int64

In [164]:
df = df_sample_check.copy()
from sklearn.preprocessing import StandardScaler

X = df[['name_score', 'location_distance', 'address_score', 'category_sim']]
y = df['is_true']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [165]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.25,
    random_state=42,
    stratify=y  # keep the same proportion of True vs False in training set and test set
)

In [166]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

gb_clf = GradientBoostingClassifier()
gb_clf.fit(X_train, y_train)

gb_y_pred = gb_clf.predict(X_test)
gb_y_prob = gb_clf.predict_proba(X_test)[:, 1]

gb_cr = classification_report(y_test, gb_y_pred)
gb_auc = roc_auc_score(y_test, gb_y_prob)

print(gb_cr)
print("GradientBoosting of AUC:", gb_auc)

              precision    recall  f1-score   support

           0       0.84      0.87      0.85        89
           1       0.97      0.96      0.97       411

    accuracy                           0.95       500
   macro avg       0.90      0.91      0.91       500
weighted avg       0.95      0.95      0.95       500

GradientBoosting of AUC: 0.9787719729899671


In [167]:
mask = ny_gplc_fsq['matched_id'].notnull()
df_pred = ny_gplc_fsq.loc[mask].copy()

feature_cols = ['name_score', 'location_distance', 'address_score', 'category_sim']

X_new = scaler.transform(df_pred[feature_cols])
df_pred['true_match_prob'] = gb_clf.predict_proba(X_new)[:, 1]
df_pred['is_true_match'] = df_pred['true_match_prob'] >= 0.5

ny_gplc_fsq.loc[mask, 'is_true_match'] = df_pred['is_true_match']
ny_gplc_fsq.loc[mask, 'true_match_prob'] = df_pred['true_match_prob']

In [168]:
ny_gplc_fsq['is_true_match'].value_counts()

is_true_match
True     43745
False    10275
Name: count, dtype: int64

In [169]:
ny_gplc_fsq.drop(columns=['cand_ids','cand_dist_m']).to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/ny_gplc_fsq.geojson')

# Google VS OSM

In [ ]:
chicago_osm = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_msa_poi/chicago_osm.geojson', driver="GeoJSON")

In [ ]:
chicago_gplc_osm = search_spatial_candidates(reference_gdf=chicago_gplc, compared_gdf=chicago_osm, k=100)

In [ ]:
chicago_gplc_osm = match_by_name(reference_gdf=chicago_gplc_osm, compared_gdf=chicago_osm, re_name_col = "name", comp_name_col = "name", comp_id_col ="cat", threshold=80)

In [ ]:
chicago_gplc_osm = address_score_check(reference_gdf=chicago_gplc_osm, compared_gdf=chicago_osm, addr_col_ref = "addr_simple", addr_col_cmp = "address", id_col = "id")

In [ ]:
chicago_gplc_osm = calculate_similarity_check(chicago_gplc_osm, cat_col_ref = "naics_definition", cat_col_cmp = "matched_cat_main", id_col = "matched_id", result_col =  "category_sim")

Encoding 21076 rows...


Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

In [ ]:
# transfer the X and Y into float type and deal with the address score
cols_to_fix = ['name_score', 'location_distance', 'address_score', 'category_sim']
for col in cols_to_fix:
    chicago_gplc_osm[col] = pd.to_numeric(chicago_gplc_osm[col], errors='coerce')
chicago_gplc_osm['address_score'] = chicago_gplc_osm['address_score'].fillna(0)
chicago_gplc_osm['category_sim'] = chicago_gplc_osm['category_sim'].fillna(0)

#### Output the matched samples

In [ ]:
df = chicago_gplc_osm[chicago_gplc_osm["matched_id"].notna()].copy()

N = 1000

weights = df["primary_cat"].map(
    df["primary_cat"].value_counts(normalize=True)
)

df_sample = df.sample(
    n=N,
    weights=weights,
    random_state=42
)
df_sample[['id','name','addr_simple','naics_definition','matched_id','matched_name','matched_address','matched_cat_main','location_distance']].to_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_google_comparison/chicago_gplc_osm_1000_sample.csv', index=False)

#### Input the matched samples

In [223]:
df_sample_check = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/ny_gplc_osm_2000_sample.csv')
df_sample_check = df_sample_check.drop(columns='location_distance')
df_sample_check = df_sample_check.merge(ny_gplc_osm[['id','name_score','location_distance','address_score','category_sim']], left_on="id", right_on="id", how="left")

In [224]:
df_sample_check['is_true'].value_counts()

is_true
1    1415
0     585
Name: count, dtype: int64

In [225]:
df = df_sample_check.copy()
from sklearn.preprocessing import StandardScaler

X = df[['name_score', 'location_distance']]
y = df['is_true']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [226]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.25,
    random_state=42,
    stratify=y  # keep the same proportion of True vs False in training set and test set
)

In [227]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

gb_clf = GradientBoostingClassifier()
gb_clf.fit(X_train, y_train)

gb_y_pred = gb_clf.predict(X_test)
gb_y_prob = gb_clf.predict_proba(X_test)[:, 1]

gb_cr = classification_report(y_test, gb_y_pred)
gb_auc = roc_auc_score(y_test, gb_y_prob)

print(gb_cr)
print("GradientBoosting of AUC:", gb_auc)

              precision    recall  f1-score   support

           0       0.84      0.95      0.89       146
           1       0.98      0.92      0.95       354

    accuracy                           0.93       500
   macro avg       0.91      0.93      0.92       500
weighted avg       0.94      0.93      0.93       500

GradientBoosting of AUC: 0.984356860924077


In [238]:
mask = ny_gplc_osm['matched_id'].notnull()
df_pred = ny_gplc_osm.loc[mask].copy()

feature_cols = ['name_score', 'location_distance']

X_new = scaler.transform(df_pred[feature_cols])
df_pred['true_match_prob'] = gb_clf.predict_proba(X_new)[:, 1]
df_pred['is_true_match'] = df_pred['true_match_prob'] >= 0.5

ny_gplc_osm.loc[mask, 'is_true_match'] = df_pred['is_true_match']
ny_gplc_osm.loc[mask, 'true_match_prob'] = df_pred['true_match_prob']

In [239]:
ny_gplc_osm['is_true_match'].value_counts()

is_true_match
True     19482
False     9574
Name: count, dtype: int64

In [242]:
ny_gplc_osm.drop(columns=['cand_ids','cand_dist_m']).to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/ny_gplc_osm.geojson')